<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/05_Dataset_and_Feature_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# 05.1 ENVIRONMENT, CONFIGURATION AND GOOGLE DRIVE
# ============================================================

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from scipy.stats import (
    skew,
    kurtosis,
    entropy
)

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

try:
    drive.mount(
        "/content/drive",
        force_remount=False
    )
except Exception as exc:
    print(
        f"Drive mount note: {exc}"
    )

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

# ------------------------------------------------------------
# DATASET REGISTRY
# ------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# ------------------------------------------------------------
# TARGET REGISTRY
# ------------------------------------------------------------

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

# ------------------------------------------------------------
# RANDOM STATE
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)

print("=" * 90)
print("NOTEBOOK 05 — DATASET AND FEATURE PROFILING")
print("=" * 90)

print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Datasets     : {DATASET_IDS}"
)

print(
    f"Targets      : {TARGET_REGISTRY}"
)

print(
    f"Random seed  : {RANDOM_SEED}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 05 — DATASET AND FEATURE PROFILING
Project root : /content/drive/MyDrive/AIR_LLM_Research
Datasets     : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets      : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Random seed  : 42


In [6]:
# ============================================================
# 05.2 PROJECT DIRECTORIES
# ============================================================

SPLIT_ROOT = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

PROFILE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "profiles"
)

DATASET_PROFILE_DIR = (
    PROFILE_ROOT /
    "dataset"
)

FEATURE_PROFILE_DIR = (
    PROFILE_ROOT /
    "feature"
)

SUMMARY_DIR = (
    PROFILE_ROOT /
    "summary"
)

for path in [
    PROFILE_ROOT,
    DATASET_PROFILE_DIR,
    FEATURE_PROFILE_DIR,
    SUMMARY_DIR
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 90)
print("PROFILING DIRECTORIES READY")
print("=" * 90)

print(
    f"Split root          : {SPLIT_ROOT}"
)

print(
    f"Profile root        : {PROFILE_ROOT}"
)

print(
    f"Dataset profile dir : {DATASET_PROFILE_DIR}"
)

print(
    f"Feature profile dir : {FEATURE_PROFILE_DIR}"
)

print(
    f"Summary dir         : {SUMMARY_DIR}"
)

PROFILING DIRECTORIES READY
Split root          : /content/drive/MyDrive/AIR_LLM_Research/data/splits
Profile root        : /content/drive/MyDrive/AIR_LLM_Research/results/profiles
Dataset profile dir : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset
Feature profile dir : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature
Summary dir         : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/summary


In [7]:
# ============================================================
# 05.3 LOAD TRAINING DATA
# ============================================================

# ------------------------------------------------------------
# DEPENDENCY CHECK
# ------------------------------------------------------------

required_config = [
    "PROJECT_ROOT",
    "SPLIT_ROOT",
    "DATASET_IDS",
    "TARGET_REGISTRY"
]

missing_config = [
    name
    for name in required_config
    if name not in globals()
]

if missing_config:
    raise RuntimeError(
        "Notebook 05 configuration is incomplete.\n"
        f"Missing objects: {missing_config}\n"
        "Run cells 05.1 and 05.2 first."
    )

# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

TRAINING_DATA = {}

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SPLIT_ROOT /
        dataset_id
    )

    X_path = (
        dataset_dir /
        "X_train.csv"
    )

    y_path = (
        dataset_dir /
        "y_train.csv"
    )

    if not X_path.exists():
        raise FileNotFoundError(
            f"Training features not found:\n{X_path}"
        )

    if not y_path.exists():
        raise FileNotFoundError(
            f"Training target not found:\n{y_path}"
        )

    X_train = pd.read_csv(
        X_path,
        low_memory=False
    )

    y_train = pd.read_csv(
        y_path,
        low_memory=False
    )

    target = TARGET_REGISTRY.get(
        dataset_id
    )

    if target is None:
        raise KeyError(
            f"No target registered for "
            f"{dataset_id}."
        )

    # --------------------------------------------------------
    # TARGET MUST NOT BE IN X
    # --------------------------------------------------------

    if target in X_train.columns:

        X_train = X_train.drop(
            columns=[target]
        )

    # --------------------------------------------------------
    # NORMALIZE TARGET COLUMN
    # --------------------------------------------------------

    if target not in y_train.columns:

        if y_train.shape[1] == 1:

            y_train = y_train.copy()

            y_train.columns = [
                target
            ]

        else:

            raise KeyError(
                f"Target '{target}' not found "
                f"in y_train for {dataset_id}."
            )

    # --------------------------------------------------------
    # RESET INDICES
    # --------------------------------------------------------

    X_train = X_train.reset_index(
        drop=True
    )

    y_train = y_train[
        [target]
    ].reset_index(
        drop=True
    )

    # --------------------------------------------------------
    # ROW VALIDATION
    # --------------------------------------------------------

    if len(X_train) != len(y_train):

        raise ValueError(
            f"Row mismatch for {dataset_id}: "
            f"X_train={len(X_train)}, "
            f"y_train={len(y_train)}"
        )

    # --------------------------------------------------------
    # COMBINE FEATURES + TARGET
    # --------------------------------------------------------

    df = pd.concat(
        [
            X_train,
            y_train
        ],
        axis=1
    )

    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' missing "
            f"after dataset construction."
        )

    if df.columns.duplicated().any():

        duplicated = (
            df.columns[
                df.columns.duplicated()
            ]
            .tolist()
        )

        raise ValueError(
            f"Duplicate columns detected "
            f"in {dataset_id}: {duplicated}"
        )

    TRAINING_DATA[
        dataset_id
    ] = df

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 90)
print("TRAINING DATA LOADED")
print("=" * 90)

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} "
        f"shape={df.shape} | "
        f"target={target}"
    )

print("=" * 90)
print("TRAINING DATA VALIDATION PASSED")
print("=" * 90)

TRAINING DATA LOADED
adult_income         shape=(19522, 15) | target=income
bank_marketing       shape=(27126, 17) | target=y
diabetes_130us       shape=(61059, 48) | target=readmitted
TRAINING DATA VALIDATION PASSED


In [8]:
# ============================================================
# 05.5 DATASET SIZE
# ============================================================

DATASET_SIZE_RECORDS = []

for dataset_id, df in TRAINING_DATA.items():

    DATASET_SIZE_RECORDS.append({

        "dataset_id":
            dataset_id,

        "n_rows":
            int(df.shape[0]),

        "n_columns":
            int(df.shape[1]),

        "memory_mb":
            round(
                float(
                    df.memory_usage(
                        deep=True
                    ).sum()
                    / (1024 ** 2)
                ),
                4
            ),

        "n_cells":
            int(
                df.shape[0] *
                df.shape[1]
            )
    })

DATASET_SIZE_DF = pd.DataFrame(
    DATASET_SIZE_RECORDS
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

required_columns = [
    "dataset_id",
    "n_rows",
    "n_columns",
    "memory_mb",
    "n_cells"
]

missing_columns = [
    column
    for column in required_columns
    if column not in DATASET_SIZE_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Dataset-size profile is missing columns: "
        f"{missing_columns}"
    )

if (
    DATASET_SIZE_DF["n_rows"] <= 0
).any():

    raise ValueError(
        "Invalid dataset size detected."
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 90)
print("05.5 DATASET SIZE PROFILE")
print("=" * 90)

display(
    DATASET_SIZE_DF
)

05.5 DATASET SIZE PROFILE


,dataset_id,n_rows,n_columns,memory_mb,n_cells
0,adult_income,19522,15,10.5805,292830
1,bank_marketing,27126,17,15.4483,461142
2,diabetes_130us,61059,48,112.8023,2930832


In [9]:
# ============================================================
# 05.6 NUMBER OF FEATURES
# ============================================================

FEATURE_COUNT_RECORDS = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # --------------------------------------------------------
    # Predictor features only
    # --------------------------------------------------------

    feature_columns = [
        column
        for column in df.columns
        if column != target
    ]

    numeric_columns = (
        df[feature_columns]
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in feature_columns
        if column not in numeric_columns
    ]

    # --------------------------------------------------------
    # Record
    # --------------------------------------------------------

    FEATURE_COUNT_RECORDS.append({

        "dataset_id":
            dataset_id,

        "n_features":
            int(
                len(feature_columns)
            ),

        "n_numeric_features":
            int(
                len(numeric_columns)
            ),

        "n_categorical_features":
            int(
                len(categorical_columns)
            ),

        "numeric_ratio":
            float(
                len(numeric_columns) /
                max(len(feature_columns), 1)
            ),

        "categorical_ratio":
            float(
                len(categorical_columns) /
                max(len(feature_columns), 1)
            )
    })

FEATURE_COUNT_DF = pd.DataFrame(
    FEATURE_COUNT_RECORDS
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if not (
    FEATURE_COUNT_DF["n_features"]
    ==
    (
        FEATURE_COUNT_DF["n_numeric_features"]
        +
        FEATURE_COUNT_DF["n_categorical_features"]
    )
).all():

    raise RuntimeError(
        "Feature-count validation failed: "
        "numeric + categorical features "
        "do not equal total features."
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 90)
print("05.6 NUMBER OF FEATURES")
print("=" * 90)

display(
    FEATURE_COUNT_DF
)

05.6 NUMBER OF FEATURES


,dataset_id,n_features,n_numeric_features,n_categorical_features,numeric_ratio,categorical_ratio
0,adult_income,14,6,8,0.428571,0.571429
1,bank_marketing,16,7,9,0.437500,0.562500
2,diabetes_130us,47,11,36,0.234043,0.765957


In [10]:
# ============================================================
# 05.7 CLASS DISTRIBUTION
# ============================================================

CLASS_DISTRIBUTION_RECORDS = []
CLASS_SUMMARY_RECORDS = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    target_series = df[
        target
    ]

    # --------------------------------------------------------
    # Class counts and proportions
    # --------------------------------------------------------

    counts = (
        target_series
        .value_counts(
            dropna=False
        )
    )

    proportions = (
        target_series
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    for class_value in counts.index:

        class_label = (
            "<MISSING>"
            if pd.isna(class_value)
            else str(class_value)
        )

        CLASS_DISTRIBUTION_RECORDS.append({

            "dataset_id":
                dataset_id,

            "target":
                target,

            "class":
                class_label,

            "count":
                int(
                    counts.loc[class_value]
                ),

            "proportion":
                float(
                    proportions.loc[class_value]
                )
        })

    # --------------------------------------------------------
    # Imbalance summary
    # --------------------------------------------------------

    non_missing_counts = (
        target_series
        .dropna()
        .value_counts()
    )

    if len(non_missing_counts) > 0:

        majority_count = int(
            non_missing_counts.max()
        )

        minority_count = int(
            non_missing_counts.min()
        )

        n_classes = int(
            non_missing_counts.shape[0]
        )

        imbalance_ratio = float(
            majority_count /
            max(minority_count, 1)
        )

    else:

        majority_count = 0
        minority_count = 0
        n_classes = 0
        imbalance_ratio = np.nan

    CLASS_SUMMARY_RECORDS.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "n_classes":
            n_classes,

        "majority_count":
            majority_count,

        "minority_count":
            minority_count,

        "imbalance_ratio":
            imbalance_ratio,

        "target_missing_count":
            int(
                target_series.isna().sum()
            ),

        "target_missing_rate":
            float(
                target_series.isna().mean()
            )
    })

# ------------------------------------------------------------
# FINAL DATAFRAMES
# ------------------------------------------------------------

CLASS_DISTRIBUTION_DF = pd.DataFrame(
    CLASS_DISTRIBUTION_RECORDS
)

CLASS_DISTRIBUTION_SUMMARY_DF = pd.DataFrame(
    CLASS_SUMMARY_RECORDS
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if CLASS_DISTRIBUTION_DF.empty:

    raise RuntimeError(
        "Class-distribution profile is empty."
    )

required_columns = [
    "dataset_id",
    "target",
    "class",
    "count",
    "proportion"
]

missing_columns = [
    column
    for column in required_columns
    if column not in CLASS_DISTRIBUTION_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Class-distribution profile is missing "
        f"columns: {missing_columns}"
    )

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("=" * 90)
print("05.7 CLASS DISTRIBUTION")
print("=" * 90)

display(
    CLASS_DISTRIBUTION_DF
)

print("=" * 90)
print("CLASS DISTRIBUTION SUMMARY")
print("=" * 90)

display(
    CLASS_DISTRIBUTION_SUMMARY_DF
)

05.7 CLASS DISTRIBUTION


,dataset_id,target,class,count,proportion
0,adult_income,income,<=50K,14819,0.759092
1,adult_income,income,>50K,4703,0.240908
2,bank_marketing,y,no,23953,0.883027
3,bank_marketing,y,yes,3173,0.116973
4,diabetes_130us,readmitted,NO,32918,0.539118
5,diabetes_130us,readmitted,>30,21327,0.349285
6,diabetes_130us,readmitted,<30,6814,0.111597


CLASS DISTRIBUTION SUMMARY


,dataset_id,target,n_classes,majority_count,minority_count,imbalance_ratio,target_missing_count,target_missing_rate
0,adult_income,income,2,14819,4703,3.150967,0,0.0
1,bank_marketing,y,2,23953,3173,7.549007,0,0.0
2,diabetes_130us,readmitted,3,32918,6814,4.830936,0,0.0


In [22]:
# ============================================================
# 05.7 NUMERICAL / CATEGORICAL RATIO
# ============================================================

TYPE_RATIO_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    X = df.drop(
        columns=[target]
    )

    n_numeric = int(
        X.select_dtypes(
            include=np.number
        ).shape[1]
    )

    n_categorical = int(
        X.select_dtypes(
            exclude=np.number
        ).shape[1]
    )

    total_features = int(
        X.shape[1]
    )

    TYPE_RATIO_PROFILE.append({

        "dataset_id":
            dataset_id,

        "numeric_count":
            n_numeric,

        "categorical_count":
            n_categorical,

        "numeric_ratio":
            float(
                n_numeric / total_features
            )
            if total_features > 0
            else 0.0,

        "categorical_ratio":
            float(
                n_categorical / total_features
            )
            if total_features > 0
            else 0.0
    })

TYPE_RATIO_DF = pd.DataFrame(
    TYPE_RATIO_PROFILE
)

TYPE_RATIO_DF = (
    TYPE_RATIO_DF
    .sort_values("dataset_id")
    .reset_index(drop=True)
)

print("=" * 90)
print("NUMERICAL / CATEGORICAL RATIO COMPLETED")
print("=" * 90)

display(
    TYPE_RATIO_DF
)

NUMERICAL / CATEGORICAL RATIO COMPLETED


,dataset_id,numeric_count,categorical_count,numeric_ratio,categorical_ratio
0,adult_income,6,8,0.428571,0.571429
1,bank_marketing,7,9,0.437500,0.562500
2,diabetes_130us,11,36,0.234043,0.765957


In [23]:
# ============================================================
# 05.9 MISSINGNESS STATISTICS
# ============================================================

MISSINGNESS_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        missing_rate = float(
            df[column].isna().mean()
        )

        MISSINGNESS_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "missing_count":
                missing_count,

            "missing_rate":
                missing_rate,

            "is_target":
                bool(column == target)
        })

MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_PROFILE
)

display(
    MISSINGNESS_DF.head(20)
)

print(
    f"Missingness profile generated: "
    f"{len(MISSINGNESS_DF):,} feature records"
)

,dataset_id,feature,missing_count,missing_rate,is_target
0,adult_income,age,0,0.0,False
1,adult_income,workclass,0,0.0,False
2,adult_income,fnlwgt,0,0.0,False
3,adult_income,education,0,0.0,False
4,adult_income,education_num,0,0.0,False
5,adult_income,marital_status,0,0.0,False
6,adult_income,occupation,0,0.0,False
7,adult_income,relationship,0,0.0,False
8,adult_income,race,0,0.0,False
9,adult_income,sex,0,0.0,False


Missingness profile generated: 80 feature records


In [24]:
# ============================================================
# 05.10 CARDINALITY
# ============================================================

CARDINALITY_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    for column in df.columns:

        non_missing = df[column].dropna()

        n_unique = int(
            non_missing.nunique()
        )

        n_rows = len(df)

        cardinality_ratio = (
            n_unique / n_rows
            if n_rows > 0
            else 0.0
        )

        CARDINALITY_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "unique_count":
                n_unique,

            "cardinality_ratio":
                float(cardinality_ratio),

            "is_constant":
                bool(n_unique <= 1),

            "is_high_cardinality":
                bool(cardinality_ratio >= 0.50)
        })

CARDINALITY_DF = pd.DataFrame(
    CARDINALITY_PROFILE
)

display(
    CARDINALITY_DF.head(20)
)

,dataset_id,feature,unique_count,cardinality_ratio,is_constant,is_high_cardinality
0,adult_income,age,73,0.003739,False,False
1,adult_income,workclass,9,0.000461,False,False
2,adult_income,fnlwgt,14799,0.758068,False,True
3,adult_income,education,16,0.000820,False,False
4,adult_income,education_num,16,0.000820,False,False
5,adult_income,marital_status,7,0.000359,False,False
6,adult_income,occupation,15,0.000768,False,False
7,adult_income,relationship,6,0.000307,False,False
8,adult_income,race,5,0.000256,False,False
9,adult_income,sex,2,0.000102,False,False


In [25]:
# ============================================================
# 05.11 DISTRIBUTION STATISTICS
# ============================================================

DISTRIBUTION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    numeric_columns = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    for column in numeric_columns:

        series = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .dropna()
        )

        if series.empty:
            continue

        skew_value = float(
            series.skew()
        )

        kurt_value = float(
            series.kurt()
        )

        DISTRIBUTION_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "mean":
                float(series.mean()),

            "median":
                float(series.median()),

            "std":
                float(series.std()),

            "min":
                float(series.min()),

            "max":
                float(series.max()),

            "skewness":
                skew_value,

            "kurtosis":
                kurt_value,

            "distribution_shape":
                (
                    "right_skewed"
                    if skew_value > 1
                    else
                    "left_skewed"
                    if skew_value < -1
                    else
                    "approximately_symmetric"
                )
        })

DISTRIBUTION_DF = pd.DataFrame(
    DISTRIBUTION_PROFILE
)

display(
    DISTRIBUTION_DF.head(20)
)

,dataset_id,feature,mean,median,std,min,max,skewness,kurtosis,distribution_shape
0,adult_income,age,38.539852,37.0,13.645484,17.0,90.0,0.558961,-0.164794,approximately_symmetric
1,adult_income,fnlwgt,190020.295205,178319.0,106163.578269,12285.0,1484705.0,1.430776,5.829256,right_skewed
2,adult_income,education_num,10.083752,10.0,2.567326,1.0,16.0,-0.297397,0.588060,approximately_symmetric
3,adult_income,capital_gain,1061.727948,0.0,7270.406604,0.0,99999.0,12.104584,159.283980,right_skewed
4,adult_income,capital_loss,87.941860,0.0,404.021716,0.0,4356.0,4.569990,20.109857,right_skewed
5,adult_income,hours_per_week,40.435867,40.0,12.401671,1.0,99.0,0.220144,2.931302,approximately_symmetric
6,bank_marketing,age,40.852393,39.0,10.636893,18.0,95.0,0.706297,0.402527,approximately_symmetric
7,bank_marketing,balance,1361.175035,449.0,3053.628721,-8019.0,102127.0,8.613740,147.921643,right_skewed
8,bank_marketing,day,15.781132,16.0,8.320549,1.0,31.0,0.098508,-1.061187,approximately_symmetric
9,bank_marketing,duration,258.993770,180.0,260.659786,0.0,4918.0,3.278843,20.597327,right_skewed


In [26]:
# ============================================================
# 05.12 OUTLIER STATISTICS
# ============================================================

OUTLIER_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    numeric_columns = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    for column in numeric_columns:

        series = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .dropna()
        )

        if series.empty:
            continue

        q1 = float(
            series.quantile(0.25)
        )

        q3 = float(
            series.quantile(0.75)
        )

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_mask = (
            (series < lower) |
            (series > upper)
        )

        outlier_count = int(
            outlier_mask.sum()
        )

        OUTLIER_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "q1":
                q1,

            "q3":
                q3,

            "iqr":
                float(iqr),

            "lower_bound":
                float(lower),

            "upper_bound":
                float(upper),

            "outlier_count":
                outlier_count,

            "outlier_rate":
                float(
                    outlier_count /
                    len(series)
                )
        })

OUTLIER_DF = pd.DataFrame(
    OUTLIER_PROFILE
)

display(
    OUTLIER_DF.head(20)
)

,dataset_id,feature,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_rate
0,adult_income,age,28.0,47.00,19.00,-0.500,75.500,138,0.007069
1,adult_income,fnlwgt,117629.5,237047.00,119417.50,-61496.750,416173.250,605,0.030991
2,adult_income,education_num,9.0,12.00,3.00,4.500,16.500,710,0.036369
3,adult_income,capital_gain,0.0,0.00,0.00,0.000,0.000,1633,0.083649
4,adult_income,capital_loss,0.0,0.00,0.00,0.000,0.000,917,0.046973
5,adult_income,hours_per_week,40.0,45.00,5.00,32.500,52.500,5416,0.277431
6,bank_marketing,age,33.0,48.00,15.00,10.500,70.500,297,0.010949
7,bank_marketing,balance,74.0,1425.00,1351.00,-1952.500,3451.500,2866,0.105655
8,bank_marketing,day,8.0,21.00,13.00,-11.500,40.500,0,0.000000
9,bank_marketing,duration,103.0,317.75,214.75,-219.125,639.875,2009,0.074062


In [27]:
# ============================================================
# 05.13 CORRELATIONS
# ============================================================

CORRELATION_PROFILE = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    numeric_df = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
    )

    if numeric_df.shape[1] >= 2:

        CORRELATION_PROFILE[
            dataset_id
        ] = numeric_df.corr(
            method="spearman"
        )

    else:

        CORRELATION_PROFILE[
            dataset_id
        ] = pd.DataFrame()

print(
    "Spearman correlation matrices constructed."
)

Spearman correlation matrices constructed.


In [28]:
# ============================================================
# 05.14 MUTUAL INFORMATION
# ============================================================

MI_SAMPLE_SIZE = 20000
MI_RANDOM_STATE = 42

MUTUAL_INFORMATION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    print(
        f"Computing mutual information: {dataset_id}"
    )

    target = TARGET_REGISTRY[dataset_id]

    X = df.drop(
        columns=[target]
    ).copy()

    y = df[target].copy()

    # --------------------------------------------------------
    # Reproducible computational sampling
    # --------------------------------------------------------

    if len(X) > MI_SAMPLE_SIZE:

        sample_idx = (
            X.sample(
                n=MI_SAMPLE_SIZE,
                random_state=MI_RANDOM_STATE
            ).index
        )

        X = X.loc[sample_idx]
        y = y.loc[sample_idx]

    # --------------------------------------------------------
    # Encode features
    # --------------------------------------------------------

    X_encoded = pd.DataFrame(
        index=X.index
    )

    discrete_features = []

    for column in X.columns:

        series = X[column]

        # Numerical feature
        if pd.api.types.is_numeric_dtype(series):

            values = pd.to_numeric(
                series,
                errors="coerce"
            )

            if values.notna().any():

                fill_value = float(
                    values.median()
                )

            else:

                fill_value = 0.0

            X_encoded[column] = (
                values.fillna(
                    fill_value
                ).astype(float)
            )

            discrete_features.append(False)

        # Categorical feature
        else:

            encoded = (
                series.astype("string")
                .fillna("__MISSING__")
                .astype("category")
                .cat.codes
            )

            X_encoded[column] = (
                encoded.astype(float)
            )

            discrete_features.append(True)

    # --------------------------------------------------------
    # Encode target
    # --------------------------------------------------------

    y_encoded = (
        y.astype("string")
        .fillna("__MISSING__")
        .astype("category")
        .cat.codes
        .to_numpy()
    )

    # --------------------------------------------------------
    # Mutual information
    # --------------------------------------------------------

    mi_values = mutual_info_classif(
        X_encoded.to_numpy(),
        y_encoded,
        discrete_features=discrete_features,
        random_state=MI_RANDOM_STATE
    )

    # --------------------------------------------------------
    # Store feature-level MI
    # --------------------------------------------------------

    dataset_mi = []

    for column, mi in zip(
        X.columns,
        mi_values
    ):

        dataset_mi.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "target":
                target,

            "mutual_information":
                float(mi)
        })

    # --------------------------------------------------------
    # Dataset-wise normalized MI
    # --------------------------------------------------------

    dataset_mi_df = pd.DataFrame(
        dataset_mi
    )

    max_mi = (
        dataset_mi_df[
            "mutual_information"
        ].max()
    )

    if (
        pd.notna(max_mi)
        and max_mi > 0
    ):

        dataset_mi_df[
            "normalized_mi"
        ] = (
            dataset_mi_df[
                "mutual_information"
            ] / max_mi
        )

    else:

        dataset_mi_df[
            "normalized_mi"
        ] = 0.0

    MUTUAL_INFORMATION_PROFILE.extend(
        dataset_mi_df.to_dict(
            orient="records"
        )
    )

MUTUAL_INFORMATION_DF = pd.DataFrame(
    MUTUAL_INFORMATION_PROFILE
)

MUTUAL_INFORMATION_DF = (
    MUTUAL_INFORMATION_DF
    .sort_values(
        [
            "dataset_id",
            "mutual_information"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

print()
print("=" * 90)
print("MUTUAL INFORMATION COMPLETED")
print("=" * 90)

print(
    f"Rows: {len(MUTUAL_INFORMATION_DF):,}"
)

print(
    f"Features evaluated: "
    f"{MUTUAL_INFORMATION_DF['feature'].nunique():,}"
)

display(
    MUTUAL_INFORMATION_DF.head(20)
)

Computing mutual information: adult_income
Computing mutual information: bank_marketing
Computing mutual information: diabetes_130us

MUTUAL INFORMATION COMPLETED
Rows: 77
Features evaluated: 73


,dataset_id,feature,target,mutual_information,normalized_mi
0,adult_income,relationship,income,0.115568,1.000000
1,adult_income,marital_status,income,0.107862,0.933318
2,adult_income,capital_gain,income,0.085868,0.743009
3,adult_income,age,income,0.072244,0.625119
4,adult_income,education_num,income,0.067522,0.584262
5,adult_income,occupation,income,0.065347,0.565440
6,adult_income,education,income,0.065343,0.565405
7,adult_income,hours_per_week,income,0.044944,0.388893
8,adult_income,capital_loss,income,0.036241,0.313586
9,adult_income,sex,income,0.027358,0.236722


In [29]:
# ============================================================
# 05.15 FEATURE DEPENDENCIES
# ============================================================

DEPENDENCY_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    features = [
        c for c in df.columns
        if c != target
    ]

    correlation_matrix = (
        CORRELATION_PROFILE.get(
            dataset_id,
            pd.DataFrame()
        )
    )

    for feature in features:

        # ----------------------------------------------------
        # Correlation-based dependency
        # ----------------------------------------------------

        max_abs_correlation = 0.0
        strongest_correlated_feature = None
        n_correlated_features = 0

        if (
            not correlation_matrix.empty
            and feature in correlation_matrix.columns
        ):

            correlations = (
                correlation_matrix[feature]
                .drop(
                    labels=[feature],
                    errors="ignore"
                )
                .dropna()
            )

            absolute_correlations = (
                correlations.abs()
            )

            # Count moderately/strongly related features
            n_correlated_features = int(
                (
                    absolute_correlations >= 0.30
                ).sum()
            )

            if not absolute_correlations.empty:

                strongest_correlated_feature = (
                    absolute_correlations.idxmax()
                )

                max_abs_correlation = float(
                    absolute_correlations.max()
                )

        # ----------------------------------------------------
        # Mutual-information dependency
        # ----------------------------------------------------

        mi_row = MUTUAL_INFORMATION_DF.loc[
            (
                MUTUAL_INFORMATION_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                MUTUAL_INFORMATION_DF[
                    "feature"
                ] == feature
            )
        ]

        if not mi_row.empty:

            mutual_information = float(
                mi_row[
                    "mutual_information"
                ].iloc[0]
            )

            normalized_mi = float(
                mi_row[
                    "normalized_mi"
                ].iloc[0]
            )

        else:

            mutual_information = 0.0
            normalized_mi = 0.0

        # ----------------------------------------------------
        # Combined dependency strength
        # ----------------------------------------------------

        dependency_strength = float(
            max(
                max_abs_correlation,
                normalized_mi
            )
        )

        # ----------------------------------------------------
        # Dependency category
        # ----------------------------------------------------

        if dependency_strength >= 0.70:

            dependency_level = "strong"

        elif dependency_strength >= 0.40:

            dependency_level = "moderate"

        elif dependency_strength >= 0.20:

            dependency_level = "weak"

        else:

            dependency_level = "low"

        # ----------------------------------------------------
        # Store profile
        # ----------------------------------------------------

        DEPENDENCY_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "max_abs_spearman":
                max_abs_correlation,

            "strongest_correlated_feature":
                strongest_correlated_feature,

            "n_correlated_features":
                n_correlated_features,

            "mutual_information":
                mutual_information,

            "normalized_mi":
                normalized_mi,

            "dependency_strength":
                dependency_strength,

            "dependency_level":
                dependency_level
        })

DEPENDENCY_DF = pd.DataFrame(
    DEPENDENCY_PROFILE
)

DEPENDENCY_DF = (
    DEPENDENCY_DF
    .sort_values(
        [
            "dataset_id",
            "dependency_strength"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

print("=" * 90)
print("FEATURE DEPENDENCY ANALYSIS COMPLETED")
print("=" * 90)

print(
    f"Dependency records: "
    f"{len(DEPENDENCY_DF):,}"
)

display(
    DEPENDENCY_DF.head(20)
)

FEATURE DEPENDENCY ANALYSIS COMPLETED
Dependency records: 77


,dataset_id,feature,max_abs_spearman,strongest_correlated_feature,n_correlated_features,mutual_information,normalized_mi,dependency_strength,dependency_level
0,adult_income,relationship,0.000000,None,0,0.115568,1.000000,1.000000,strong
1,adult_income,marital_status,0.000000,None,0,0.107862,0.933318,0.933318,strong
2,adult_income,capital_gain,0.125391,age,0,0.085868,0.743009,0.743009,strong
3,adult_income,age,0.141710,hours_per_week,0,0.072244,0.625119,0.625119,moderate
4,adult_income,education_num,0.171823,hours_per_week,0,0.067522,0.584262,0.584262,moderate
5,adult_income,occupation,0.000000,None,0,0.065347,0.565440,0.565440,moderate
6,adult_income,education,0.000000,None,0,0.065343,0.565405,0.565405,moderate
7,adult_income,hours_per_week,0.171823,education_num,0,0.044944,0.388893,0.388893,weak
8,adult_income,capital_loss,0.066966,capital_gain,0,0.036241,0.313586,0.313586,weak
9,adult_income,sex,0.000000,None,0,0.027358,0.236722,0.236722,weak


In [30]:
# ============================================================
# 05.16 COMPUTATIONAL SCALE
# ============================================================

COMPUTATIONAL_SCALE_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    # --------------------------------------------------------
    # Basic dimensions
    # --------------------------------------------------------

    n_rows = int(len(df))

    feature_df = df.drop(
        columns=[target]
    )

    n_features = int(
        feature_df.shape[1]
    )

    n_numeric = int(
        feature_df
        .select_dtypes(
            include=np.number
        )
        .shape[1]
    )

    n_categorical = int(
        n_features - n_numeric
    )

    # --------------------------------------------------------
    # Cell counts
    # --------------------------------------------------------

    total_cells = int(
        n_rows * n_features
    )

    missing_cells = int(
        feature_df
        .isna()
        .sum()
        .sum()
    )

    missing_cell_rate = (
        float(
            missing_cells /
            total_cells
        )
        if total_cells > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Memory usage
    # --------------------------------------------------------

    memory_mb = float(
        df.memory_usage(
            deep=True
        ).sum()
        / (1024 ** 2)
    )

    # --------------------------------------------------------
    # Dataset scale
    # --------------------------------------------------------

    if total_cells < 1_000_000:

        scale_category = "small"

    elif total_cells < 10_000_000:

        scale_category = "medium"

    else:

        scale_category = "large"

    # --------------------------------------------------------
    # Row-scale category
    # --------------------------------------------------------

    if n_rows < 10_000:

        row_scale_category = "low"

    elif n_rows < 100_000:

        row_scale_category = "moderate"

    else:

        row_scale_category = "high"

    # --------------------------------------------------------
    # Feature-scale category
    # --------------------------------------------------------

    if n_features < 20:

        feature_scale_category = "low"

    elif n_features < 100:

        feature_scale_category = "moderate"

    else:

        feature_scale_category = "high"

    # --------------------------------------------------------
    # Store computational profile
    # --------------------------------------------------------

    COMPUTATIONAL_SCALE_PROFILE.append({

        "dataset_id":
            dataset_id,

        "rows":
            n_rows,

        "features":
            n_features,

        "numeric_features":
            n_numeric,

        "categorical_features":
            n_categorical,

        "cells":
            total_cells,

        "missing_cells":
            missing_cells,

        "missing_cell_rate":
            missing_cell_rate,

        "memory_mb":
            memory_mb,

        "scale_category":
            scale_category,

        "row_scale_category":
            row_scale_category,

        "feature_scale_category":
            feature_scale_category
    })

COMPUTATIONAL_SCALE_DF = pd.DataFrame(
    COMPUTATIONAL_SCALE_PROFILE
)

COMPUTATIONAL_SCALE_DF = (
    COMPUTATIONAL_SCALE_DF
    .sort_values(
        "dataset_id"
    )
    .reset_index(drop=True)
)

print("=" * 90)
print("COMPUTATIONAL SCALE PROFILE COMPLETED")
print("=" * 90)

display(
    COMPUTATIONAL_SCALE_DF
)

COMPUTATIONAL SCALE PROFILE COMPLETED


,dataset_id,rows,features,numeric_features,categorical_features,cells,missing_cells,missing_cell_rate,memory_mb,scale_category,row_scale_category,feature_scale_category
0,adult_income,19522,14,6,8,273308,0,0.00000,10.580482,small,moderate,low
1,bank_marketing,27126,16,7,9,434016,0,0.00000,15.448343,small,moderate,low
2,diabetes_130us,61059,47,11,36,2869773,224503,0.07823,112.802345,medium,moderate,moderate


In [31]:
# ============================================================
# 05.17 DATASET PROFILE CONSTRUCTION
# ============================================================

DATASET_PROFILES = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    X = df.drop(
        columns=[target]
    )

    # --------------------------------------------------------
    # Retrieve previously constructed profiles
    # --------------------------------------------------------

    size_row = (
        DATASET_SIZE_DF[
            DATASET_SIZE_DF["dataset_id"] == dataset_id
        ]
        .iloc[0]
    )

    feature_count_row = (
        FEATURE_COUNT_DF[
            FEATURE_COUNT_DF["dataset_id"] == dataset_id
        ]
        .iloc[0]
    )

    type_row = (
        TYPE_RATIO_DF[
            TYPE_RATIO_DF["dataset_id"] == dataset_id
        ]
        .iloc[0]
    )

    scale_row = (
        COMPUTATIONAL_SCALE_DF[
            COMPUTATIONAL_SCALE_DF["dataset_id"] == dataset_id
        ]
        .iloc[0]
    )

    # --------------------------------------------------------
    # Missingness summary
    # --------------------------------------------------------

    feature_missingness = (
        MISSINGNESS_DF[
            MISSINGNESS_DF["dataset_id"] == dataset_id
        ]
        .copy()
    )

    feature_missingness = (
        feature_missingness[
            feature_missingness["feature"] != target
        ]
    )

    dataset_missing_rate = float(
        X.isna().mean().mean()
    )

    incomplete_feature_count = int(
        (
            feature_missingness["missing_count"] > 0
        ).sum()
    )

    # --------------------------------------------------------
    # Cardinality summary
    # --------------------------------------------------------

    cardinality_df = (
        CARDINALITY_DF[
            CARDINALITY_DF["dataset_id"] == dataset_id
        ]
        .copy()
    )

    cardinality_df = (
        cardinality_df[
            cardinality_df["feature"] != target
        ]
    )

    high_cardinality_count = int(
        cardinality_df[
            "is_high_cardinality"
        ].sum()
    )

    # --------------------------------------------------------
    # Distribution statistics
    # --------------------------------------------------------

    distribution_records = (
        DISTRIBUTION_DF[
            DISTRIBUTION_DF["dataset_id"] == dataset_id
        ]
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # Outlier statistics
    # --------------------------------------------------------

    outlier_records = (
        OUTLIER_DF[
            OUTLIER_DF["dataset_id"] == dataset_id
        ]
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # Mutual information
    # --------------------------------------------------------

    mi_records = (
        MUTUAL_INFORMATION_DF[
            MUTUAL_INFORMATION_DF["dataset_id"] == dataset_id
        ]
        .sort_values(
            "mutual_information",
            ascending=False
        )
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # Feature dependency information
    # --------------------------------------------------------

    dependency_records = (
        DEPENDENCY_DF[
            DEPENDENCY_DF["dataset_id"] == dataset_id
        ]
        .sort_values(
            "dependency_strength",
            ascending=False
        )
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # Target distribution
    # --------------------------------------------------------

    target_distribution = (
        CLASS_DISTRIBUTION_DF[
            CLASS_DISTRIBUTION_DF["dataset_id"] == dataset_id
        ]
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # Dataset-level profile
    # --------------------------------------------------------

    DATASET_PROFILES[dataset_id] = {

        "dataset_id":
            dataset_id,

        # ----------------------------------------------------
        # Size
        # ----------------------------------------------------

        "size": {

            "rows":
                int(size_row["n_rows"]),

            "columns":
                int(size_row["n_columns"]),

            "memory_mb":
                float(size_row["memory_mb"])
        },

        # ----------------------------------------------------
        # Feature composition
        # ----------------------------------------------------

        "type": {

            "n_features":
                int(
                    feature_count_row[
                        "n_features"
                    ]
                ),

            "n_numeric_features":
                int(
                    feature_count_row[
                        "n_numeric_features"
                    ]
                ),

            "n_categorical_features":
                int(
                    feature_count_row[
                        "n_categorical_features"
                    ]
                ),

            "numeric_count":
                int(
                    type_row[
                        "numeric_count"
                    ]
                ),

            "categorical_count":
                int(
                    type_row[
                        "categorical_count"
                    ]
                ),

            "numeric_ratio":
                float(
                    type_row[
                        "numeric_ratio"
                    ]
                ),

            "categorical_ratio":
                float(
                    type_row[
                        "categorical_ratio"
                    ]
                )
        },

        # ----------------------------------------------------
        # Missingness
        # ----------------------------------------------------

        "missingness": {

            "overall_feature_missing_rate":
                dataset_missing_rate,

            "incomplete_feature_count":
                incomplete_feature_count,

            "feature_statistics":
                feature_missingness.to_dict(
                    orient="records"
                )
        },

        # ----------------------------------------------------
        # Cardinality
        # ----------------------------------------------------

        "cardinality": {

            "high_cardinality_feature_count":
                high_cardinality_count,

            "feature_statistics":
                cardinality_df.to_dict(
                    orient="records"
                )
        },

        # ----------------------------------------------------
        # Distribution
        # ----------------------------------------------------

        "distribution": {

            "numeric_features":
                distribution_records,

            "outlier_statistics":
                outlier_records
        },

        # ----------------------------------------------------
        # Dependency
        # ----------------------------------------------------

        "dependency": {

            "mutual_information":
                mi_records,

            "feature_dependencies":
                dependency_records
        },

        # ----------------------------------------------------
        # Task
        # ----------------------------------------------------

        "task": {

            "target":
                target,

            "target_distribution":
                target_distribution
        },

        # ----------------------------------------------------
        # Computational scale
        # ----------------------------------------------------

        "computational_scale":
            scale_row.to_dict()
    }

print("=" * 90)
print("DATASET PROFILES CONSTRUCTED")
print("=" * 90)

for dataset_id, profile in DATASET_PROFILES.items():

    print(
        f"{dataset_id:20s} | "
        f"Rows={profile['size']['rows']:,} | "
        f"Features={profile['type']['n_features']} | "
        f"Numeric={profile['type']['n_numeric_features']} | "
        f"Categorical={profile['type']['n_categorical_features']} | "
        f"Incomplete={profile['missingness']['incomplete_feature_count']}"
    )

DATASET PROFILES CONSTRUCTED
adult_income         | Rows=19,522 | Features=14 | Numeric=6 | Categorical=8 | Incomplete=0
bank_marketing       | Rows=27,126 | Features=16 | Numeric=7 | Categorical=9 | Incomplete=0
diabetes_130us       | Rows=61,059 | Features=47 | Numeric=11 | Categorical=36 | Incomplete=9


In [32]:
# ============================================================
# 05.X PROFILE OBJECT AVAILABILITY CHECK
# ============================================================

REQUIRED_PROFILE_OBJECTS = [
    "TRAINING_DATA",
    "TARGET_REGISTRY",
    "DATASET_SIZE_DF",
    "FEATURE_COUNT_DF",
    "TYPE_RATIO_DF",
    "CLASS_DISTRIBUTION_DF",
    "MISSINGNESS_DF",
    "CARDINALITY_DF",
    "DISTRIBUTION_DF",
    "OUTLIER_DF",
    "CORRELATION_PROFILE",
    "MUTUAL_INFORMATION_DF",
    "DEPENDENCY_DF",
    "COMPUTATIONAL_SCALE_DF"
]

print("=" * 90)
print("NOTEBOOK 05 — PROFILE OBJECT AVAILABILITY")
print("=" * 90)

MISSING_OBJECTS = []

for object_name in REQUIRED_PROFILE_OBJECTS:

    available = object_name in globals()

    print(
        f"{object_name:30s} : "
        f"{'AVAILABLE' if available else 'MISSING'}"
    )

    if not available:
        MISSING_OBJECTS.append(
            object_name
        )

print()

if MISSING_OBJECTS:

    print("Missing objects:")

    for object_name in MISSING_OBJECTS:
        print(
            f"  - {object_name}"
        )

else:

    print(
        "All Notebook 05 profile objects are available."
    )

NOTEBOOK 05 — PROFILE OBJECT AVAILABILITY
TRAINING_DATA                  : AVAILABLE
TARGET_REGISTRY                : AVAILABLE
DATASET_SIZE_DF                : AVAILABLE
FEATURE_COUNT_DF               : AVAILABLE
TYPE_RATIO_DF                  : AVAILABLE
CLASS_DISTRIBUTION_DF          : AVAILABLE
MISSINGNESS_DF                 : AVAILABLE
CARDINALITY_DF                 : AVAILABLE
DISTRIBUTION_DF                : AVAILABLE
OUTLIER_DF                     : AVAILABLE
CORRELATION_PROFILE            : AVAILABLE
MUTUAL_INFORMATION_DF          : AVAILABLE
DEPENDENCY_DF                  : AVAILABLE
COMPUTATIONAL_SCALE_DF         : AVAILABLE

All Notebook 05 profile objects are available.


In [33]:
# ============================================================
# 05.18 FEATURE TYPE
# ============================================================

FEATURE_TYPE_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    for column in df.columns:

        is_target = column == target

        if is_target:

            feature_role = "target"

        elif pd.api.types.is_numeric_dtype(
            df[column]
        ):

            feature_role = "numerical"

        else:

            feature_role = "categorical"

        FEATURE_TYPE_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "feature_role":
                feature_role,

            "dtype":
                str(df[column].dtype),

            "is_numeric":
                bool(
                    feature_role == "numerical"
                ),

            "is_categorical":
                bool(
                    feature_role == "categorical"
                ),

            "is_target":
                bool(is_target)
        })

FEATURE_TYPE_DF = pd.DataFrame(
    FEATURE_TYPE_PROFILE
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert not FEATURE_TYPE_DF.empty, (
    "FEATURE_TYPE_DF is empty."
)

assert not FEATURE_TYPE_DF[
    ["dataset_id", "feature"]
].duplicated().any(), (
    "Duplicate dataset-feature combinations "
    "found in FEATURE_TYPE_DF."
)

assert (
    FEATURE_TYPE_DF["is_target"]
    ==
    (
        FEATURE_TYPE_DF["feature_role"]
        == "target"
    )
).all(), (
    "Target-role consistency check failed."
)

assert (
    FEATURE_TYPE_DF["is_numeric"]
    ==
    (
        FEATURE_TYPE_DF["feature_role"]
        == "numerical"
    )
).all(), (
    "Numerical-role consistency check failed."
)

assert (
    FEATURE_TYPE_DF["is_categorical"]
    ==
    (
        FEATURE_TYPE_DF["feature_role"]
        == "categorical"
    )
).all(), (
    "Categorical-role consistency check failed."
)

print("=" * 90)
print("FEATURE TYPE PROFILE")
print("=" * 90)

print(
    f"Rows : {len(FEATURE_TYPE_DF):,}"
)

print(
    f"Datasets : "
    f"{FEATURE_TYPE_DF['dataset_id'].nunique()}"
)

print(
    f"Numerical : "
    f"{FEATURE_TYPE_DF['is_numeric'].sum():,}"
)

print(
    f"Categorical : "
    f"{FEATURE_TYPE_DF['is_categorical'].sum():,}"
)

print(
    f"Targets : "
    f"{FEATURE_TYPE_DF['is_target'].sum():,}"
)

display(
    FEATURE_TYPE_DF.head(20)
)

FEATURE TYPE PROFILE
Rows : 80
Datasets : 3
Numerical : 24
Categorical : 53
Targets : 3


,dataset_id,feature,feature_role,dtype,is_numeric,is_categorical,is_target
0,adult_income,age,numerical,int64,True,False,False
1,adult_income,workclass,categorical,object,False,True,False
2,adult_income,fnlwgt,numerical,int64,True,False,False
3,adult_income,education,categorical,object,False,True,False
4,adult_income,education_num,numerical,int64,True,False,False
5,adult_income,marital_status,categorical,object,False,True,False
6,adult_income,occupation,categorical,object,False,True,False
7,adult_income,relationship,categorical,object,False,True,False
8,adult_income,race,categorical,object,False,True,False
9,adult_income,sex,categorical,object,False,True,False


In [34]:
# ============================================================
# 05.19 FEATURE MISSINGNESS
# ============================================================

FEATURE_MISSINGNESS_DF = (
    MISSINGNESS_DF[
        [
            "dataset_id",
            "feature",
            "missing_count",
            "missing_rate",
            "is_target"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# MISSINGNESS CATEGORY
# ------------------------------------------------------------

FEATURE_MISSINGNESS_DF[
    "missingness_category"
] = pd.cut(
    FEATURE_MISSINGNESS_DF["missing_rate"],
    bins=[
        -np.inf,
        0.0,
        0.10,
        0.30,
        0.50,
        np.inf
    ],
    labels=[
        "none",
        "low",
        "moderate",
        "high",
        "very_high"
    ],
    include_lowest=True,
    right=True
)

# ------------------------------------------------------------
# NUMERIC SAFETY
# ------------------------------------------------------------

FEATURE_MISSINGNESS_DF[
    "missing_count"
] = (
    pd.to_numeric(
        FEATURE_MISSINGNESS_DF[
            "missing_count"
        ],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

FEATURE_MISSINGNESS_DF[
    "missing_rate"
] = (
    pd.to_numeric(
        FEATURE_MISSINGNESS_DF[
            "missing_rate"
        ],
        errors="coerce"
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert not FEATURE_MISSINGNESS_DF.empty, (
    "FEATURE_MISSINGNESS_DF is empty."
)

assert not FEATURE_MISSINGNESS_DF[
    ["dataset_id", "feature"]
].duplicated().any(), (
    "Duplicate dataset-feature combinations "
    "found in FEATURE_MISSINGNESS_DF."
)

assert (
    FEATURE_MISSINGNESS_DF["missing_rate"]
    .between(0.0, 1.0)
    .all()
), (
    "Invalid missing_rate detected."
)

assert (
    FEATURE_MISSINGNESS_DF["missing_count"]
    >= 0
).all(), (
    "Negative missing_count detected."
)

# ------------------------------------------------------------
# ORDER COLUMNS
# ------------------------------------------------------------

FEATURE_MISSINGNESS_DF = (
    FEATURE_MISSINGNESS_DF[
        [
            "dataset_id",
            "feature",
            "missing_count",
            "missing_rate",
            "missingness_category",
            "is_target"
        ]
    ]
)

print("=" * 90)
print("FEATURE MISSINGNESS PROFILE")
print("=" * 90)

print(
    f"Rows       : "
    f"{len(FEATURE_MISSINGNESS_DF):,}"
)

print(
    f"Incomplete : "
    f"{(
        FEATURE_MISSINGNESS_DF["missing_rate"] > 0
    ).sum():,}"
)

print(
    f"Complete   : "
    f"{(
        FEATURE_MISSINGNESS_DF["missing_rate"] == 0
    ).sum():,}"
)

print(
    f"Targets    : "
    f"{FEATURE_MISSINGNESS_DF["is_target"].sum():,}"
)

display(
    FEATURE_MISSINGNESS_DF.head(20)
)

FEATURE MISSINGNESS PROFILE
Rows       : 80
Incomplete : 9
Complete   : 71
Targets    : 3


,dataset_id,feature,missing_count,missing_rate,missingness_category,is_target
0,adult_income,age,0,0.0,none,False
1,adult_income,workclass,0,0.0,none,False
2,adult_income,fnlwgt,0,0.0,none,False
3,adult_income,education,0,0.0,none,False
4,adult_income,education_num,0,0.0,none,False
5,adult_income,marital_status,0,0.0,none,False
6,adult_income,occupation,0,0.0,none,False
7,adult_income,relationship,0,0.0,none,False
8,adult_income,race,0,0.0,none,False
9,adult_income,sex,0,0.0,none,False


In [35]:
# ============================================================
# 05.20 FEATURE DISTRIBUTION
# ============================================================

FEATURE_DISTRIBUTION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    for column in df.columns:

        # ----------------------------------------------------
        # TARGET IS NOT AN IMPUTATION CANDIDATE
        # ----------------------------------------------------

        if column == target:
            continue

        series = df[column]

        # ====================================================
        # NUMERICAL FEATURE
        # ====================================================

        if pd.api.types.is_numeric_dtype(series):

            values = (
                pd.to_numeric(
                    series,
                    errors="coerce"
                )
                .dropna()
            )

            if values.empty:
                continue

            skew_value = float(
                values.skew()
            )

            kurt_value = float(
                values.kurt()
            )

            if skew_value > 1:
                distribution_shape = "right_skewed"

            elif skew_value < -1:
                distribution_shape = "left_skewed"

            else:
                distribution_shape = (
                    "approximately_symmetric"
                )

            FEATURE_DISTRIBUTION_PROFILE.append({

                "dataset_id":
                    dataset_id,

                "feature":
                    column,

                "distribution_type":
                    "numerical",

                "mean":
                    float(values.mean()),

                "median":
                    float(values.median()),

                "std":
                    float(values.std()),

                "min":
                    float(values.min()),

                "max":
                    float(values.max()),

                "skewness":
                    skew_value,

                "kurtosis":
                    kurt_value,

                "distribution_shape":
                    distribution_shape,

                "top_category_frequency":
                    np.nan
            })

        # ====================================================
        # CATEGORICAL FEATURE
        # ====================================================

        else:

            counts = (
                series
                .astype("string")
                .fillna("__MISSING__")
                .value_counts(
                    normalize=True,
                    dropna=False
                )
            )

            top_frequency = (
                float(counts.iloc[0])
                if not counts.empty
                else 0.0
            )

            FEATURE_DISTRIBUTION_PROFILE.append({

                "dataset_id":
                    dataset_id,

                "feature":
                    column,

                "distribution_type":
                    "categorical",

                "mean":
                    np.nan,

                "median":
                    np.nan,

                "std":
                    np.nan,

                "min":
                    np.nan,

                "max":
                    np.nan,

                "skewness":
                    np.nan,

                "kurtosis":
                    np.nan,

                "distribution_shape":
                    "categorical",

                "top_category_frequency":
                    top_frequency
            })

FEATURE_DISTRIBUTION_DF = pd.DataFrame(
    FEATURE_DISTRIBUTION_PROFILE
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert not FEATURE_DISTRIBUTION_DF.empty, (
    "FEATURE_DISTRIBUTION_DF is empty."
)

assert not FEATURE_DISTRIBUTION_DF[
    ["dataset_id", "feature"]
].duplicated().any(), (
    "Duplicate dataset-feature combinations "
    "found in FEATURE_DISTRIBUTION_DF."
)

assert set(
    FEATURE_DISTRIBUTION_DF[
        "distribution_type"
    ].dropna().unique()
).issubset({
    "numerical",
    "categorical"
}), (
    "Unexpected distribution type detected."
)

# Numerical features must have numerical statistics.
numeric_rows = (
    FEATURE_DISTRIBUTION_DF[
        "distribution_type"
    ] == "numerical"
)

assert (
    FEATURE_DISTRIBUTION_DF.loc[
        numeric_rows,
        "mean"
    ].notna()
).all(), (
    "Numerical features contain missing means."
)

# Categorical frequencies must be valid.
categorical_rows = (
    FEATURE_DISTRIBUTION_DF[
        "distribution_type"
    ] == "categorical"
)

assert (
    FEATURE_DISTRIBUTION_DF.loc[
        categorical_rows,
        "top_category_frequency"
    ]
    .between(0.0, 1.0)
    .all()
), (
    "Invalid categorical frequency detected."
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 90)
print("FEATURE DISTRIBUTION PROFILE")
print("=" * 90)

print(
    f"Total features : "
    f"{len(FEATURE_DISTRIBUTION_DF):,}"
)

print(
    f"Numerical      : "
    f"{numeric_rows.sum():,}"
)

print(
    f"Categorical    : "
    f"{categorical_rows.sum():,}"
)

display(
    FEATURE_DISTRIBUTION_DF.head(20)
)

FEATURE DISTRIBUTION PROFILE
Total features : 77
Numerical      : 24
Categorical    : 53


,dataset_id,feature,distribution_type,mean,median,std,min,max,skewness,kurtosis,distribution_shape,top_category_frequency
0,adult_income,age,numerical,38.539852,37.0,13.645484,17.0,90.0,0.558961,-0.164794,approximately_symmetric,NaN
1,adult_income,workclass,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.693576
2,adult_income,fnlwgt,numerical,190020.295205,178319.0,106163.578269,12285.0,1484705.0,1.430776,5.829256,right_skewed,NaN
3,adult_income,education,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.323328
4,adult_income,education_num,numerical,10.083752,10.0,2.567326,1.0,16.0,-0.297397,0.588060,approximately_symmetric,NaN
5,adult_income,marital_status,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.459379
6,adult_income,occupation,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.127753
7,adult_income,relationship,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.404979
8,adult_income,race,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.856879
9,adult_income,sex,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,categorical,0.668528


In [36]:
# ============================================================
# 05.22 FEATURE DEPENDENCIES
# ============================================================

FEATURE_DEPENDENCY_DF = (
    DEPENDENCY_DF[
        [
            "dataset_id",
            "feature",
            "max_abs_spearman",
            "strongest_correlated_feature",
            "mutual_information"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# NUMERIC SAFETY
# ------------------------------------------------------------

FEATURE_DEPENDENCY_DF[
    "max_abs_spearman"
] = (
    pd.to_numeric(
        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ],
        errors="coerce"
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)

FEATURE_DEPENDENCY_DF[
    "mutual_information"
] = (
    pd.to_numeric(
        FEATURE_DEPENDENCY_DF[
            "mutual_information"
        ],
        errors="coerce"
    )
    .fillna(0.0)
    .clip(lower=0.0)
)

# ------------------------------------------------------------
# CORRELATION-BASED DEPENDENCY STRENGTH
# ------------------------------------------------------------

FEATURE_DEPENDENCY_DF[
    "dependency_strength"
] = np.select(

    [
        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.70,

        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.40,

        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.20
    ],

    [
        "strong",
        "moderate",
        "weak"
    ],

    default="very_weak"
)

# ------------------------------------------------------------
# DEPENDENCY AVAILABILITY
# ------------------------------------------------------------

FEATURE_DEPENDENCY_DF[
    "has_correlated_predictor"
] = (
    FEATURE_DEPENDENCY_DF[
        "strongest_correlated_feature"
    ]
    .notna()
    &
    FEATURE_DEPENDENCY_DF[
        "strongest_correlated_feature"
    ]
    .astype(str)
    .ne("")
)

FEATURE_DEPENDENCY_DF[
    "has_predictive_dependency"
] = (
    (
        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.20
    )
    |
    (
        FEATURE_DEPENDENCY_DF[
            "mutual_information"
        ] > 0.0
    )
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert not FEATURE_DEPENDENCY_DF.empty, (
    "FEATURE_DEPENDENCY_DF is empty."
)

assert not FEATURE_DEPENDENCY_DF[
    ["dataset_id", "feature"]
].duplicated().any(), (
    "Duplicate dataset-feature combinations "
    "found in FEATURE_DEPENDENCY_DF."
)

assert (
    FEATURE_DEPENDENCY_DF[
        "max_abs_spearman"
    ]
    .between(0.0, 1.0)
    .all()
), (
    "Invalid Spearman dependency detected."
)

assert (
    FEATURE_DEPENDENCY_DF[
        "mutual_information"
    ]
    >= 0
).all(), (
    "Negative mutual-information value detected."
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 90)
print("FEATURE DEPENDENCY PROFILE")
print("=" * 90)

print(
    f"Features                 : "
    f"{len(FEATURE_DEPENDENCY_DF):,}"
)

print(
    f"Strong dependencies     : "
    f"{(
        FEATURE_DEPENDENCY_DF[
            'dependency_strength'
        ] == 'strong'
    ).sum():,}"
)

print(
    f"Moderate dependencies   : "
    f"{(
        FEATURE_DEPENDENCY_DF[
            'dependency_strength'
        ] == 'moderate'
    ).sum():,}"
)

print(
    f"Weak dependencies       : "
    f"{(
        FEATURE_DEPENDENCY_DF[
            'dependency_strength'
        ] == 'weak'
    ).sum():,}"
)

print(
    f"Predictive dependencies : "
    f"{FEATURE_DEPENDENCY_DF[
        'has_predictive_dependency'
    ].sum():,}"
)

display(
    FEATURE_DEPENDENCY_DF.head(20)
)

FEATURE DEPENDENCY PROFILE
Features                 : 77
Strong dependencies     : 2
Moderate dependencies   : 2
Weak dependencies       : 8
Predictive dependencies : 69


,dataset_id,feature,max_abs_spearman,strongest_correlated_feature,mutual_information,dependency_strength,has_correlated_predictor,has_predictive_dependency
0,adult_income,relationship,0.000000,None,0.115568,very_weak,False,True
1,adult_income,marital_status,0.000000,None,0.107862,very_weak,False,True
2,adult_income,capital_gain,0.125391,age,0.085868,very_weak,True,True
3,adult_income,age,0.141710,hours_per_week,0.072244,very_weak,True,True
4,adult_income,education_num,0.171823,hours_per_week,0.067522,very_weak,True,True
5,adult_income,occupation,0.000000,None,0.065347,very_weak,False,True
6,adult_income,education,0.000000,None,0.065343,very_weak,False,True
7,adult_income,hours_per_week,0.171823,education_num,0.044944,very_weak,True,True
8,adult_income,capital_loss,0.066966,capital_gain,0.036241,very_weak,True,True
9,adult_income,sex,0.000000,None,0.027358,very_weak,False,True


In [37]:
# ============================================================
# 05.24 FEATURE PROFILE CONSTRUCTION
# ============================================================

FEATURE_PROFILES = {}

FEATURE_PROFILE_VALIDATION = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    FEATURE_PROFILES[dataset_id] = {}

    # --------------------------------------------------------
    # ONLY FEATURES RELEVANT TO IMPUTATION
    # --------------------------------------------------------

    features = [
        column
        for column in df.columns
        if column != target
    ]

    for feature in features:

        # ====================================================
        # T — FEATURE TYPE
        # ====================================================

        type_row = FEATURE_TYPE_DF.loc[
            (
                FEATURE_TYPE_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                FEATURE_TYPE_DF["feature"]
                == feature
            )
        ]

        # ====================================================
        # M — FEATURE MISSINGNESS
        # ====================================================

        missing_row = FEATURE_MISSINGNESS_DF.loc[
            (
                FEATURE_MISSINGNESS_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                FEATURE_MISSINGNESS_DF["feature"]
                == feature
            )
        ]

        # ====================================================
        # D — FEATURE DISTRIBUTION
        # ====================================================

        distribution_row = (
            FEATURE_DISTRIBUTION_DF.loc[
                (
                    FEATURE_DISTRIBUTION_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    FEATURE_DISTRIBUTION_DF["feature"]
                    == feature
                )
            ]
        )

        # ====================================================
        # C — FEATURE CARDINALITY
        # ====================================================

        cardinality_row = CARDINALITY_DF.loc[
            (
                CARDINALITY_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                CARDINALITY_DF["feature"]
                == feature
            )
        ]

        # ====================================================
        # Y — FEATURE DEPENDENCIES
        # ====================================================

        dependency_row = (
            FEATURE_DEPENDENCY_DF.loc[
                (
                    FEATURE_DEPENDENCY_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    FEATURE_DEPENDENCY_DF["feature"]
                    == feature
                )
            ]
        )

        # ====================================================
        # R — PREDICTIVE RELEVANCE
        # ====================================================

        relevance_row = (
            MUTUAL_INFORMATION_DF.loc[
                (
                    MUTUAL_INFORMATION_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    MUTUAL_INFORMATION_DF["feature"]
                    == feature
                )
            ]
        )

        # ----------------------------------------------------
        # COMPONENT AVAILABILITY
        # ----------------------------------------------------

        component_status = {

            "T":
                not type_row.empty,

            "M":
                not missing_row.empty,

            "D":
                not distribution_row.empty,

            "R":
                not relevance_row.empty,

            "C":
                not cardinality_row.empty,

            "Y":
                not dependency_row.empty
        }

        # ----------------------------------------------------
        # CONSTRUCT FEATURE PROFILE
        # ----------------------------------------------------

        feature_profile = {

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "is_target":
                False,

            "T":
                (
                    type_row.iloc[0].to_dict()
                    if not type_row.empty
                    else {}
                ),

            "M":
                (
                    missing_row.iloc[0].to_dict()
                    if not missing_row.empty
                    else {}
                ),

            "D":
                (
                    distribution_row.iloc[0].to_dict()
                    if not distribution_row.empty
                    else {}
                ),

            "R":
                (
                    relevance_row.iloc[0].to_dict()
                    if not relevance_row.empty
                    else {}
                ),

            "C":
                (
                    cardinality_row.iloc[0].to_dict()
                    if not cardinality_row.empty
                    else {}
                ),

            "Y":
                (
                    dependency_row.iloc[0].to_dict()
                    if not dependency_row.empty
                    else {}
                )
        }

        FEATURE_PROFILES[
            dataset_id
        ][
            feature
        ] = feature_profile

        # ----------------------------------------------------
        # VALIDATION RECORD
        # ----------------------------------------------------

        FEATURE_PROFILE_VALIDATION.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "type_available":
                component_status["T"],

            "missingness_available":
                component_status["M"],

            "distribution_available":
                component_status["D"],

            "relevance_available":
                component_status["R"],

            "cardinality_available":
                component_status["C"],

            "dependency_available":
                component_status["Y"],

            "complete_profile":
                all(
                    component_status.values()
                )
        })

FEATURE_PROFILE_VALIDATION_DF = pd.DataFrame(
    FEATURE_PROFILE_VALIDATION
)

# ============================================================
# VALIDATION
# ============================================================

assert FEATURE_PROFILES, (
    "FEATURE_PROFILES is empty."
)

assert not FEATURE_PROFILE_VALIDATION_DF.empty, (
    "FEATURE_PROFILE_VALIDATION_DF is empty."
)

# Every feature must have T, M and C.
required_components = [
    "type_available",
    "missingness_available",
    "cardinality_available"
]

for component in required_components:

    assert (
        FEATURE_PROFILE_VALIDATION_DF[
            component
        ]
        .all()
    ), (
        f"Required feature-profile "
        f"component missing: {component}"
    )

# ------------------------------------------------------------
# REPORT OPTIONAL COMPONENTS
# ------------------------------------------------------------

print("=" * 90)
print("FEATURE PROFILE CONSTRUCTION")
print("=" * 90)

print(
    f"Datasets : "
    f"{len(FEATURE_PROFILES)}"
)

print(
    f"Features : "
    f"{len(FEATURE_PROFILE_VALIDATION_DF):,}"
)

print(
    f"Complete profiles : "
    f"{FEATURE_PROFILE_VALIDATION_DF[
        'complete_profile'
    ].sum():,}"
)

print(
    f"Incomplete profiles : "
    f"{(
        ~FEATURE_PROFILE_VALIDATION_DF[
            'complete_profile'
        ]
    ).sum():,}"
)

display(
    FEATURE_PROFILE_VALIDATION_DF.head(20)
)

print()
print(
    "Feature profiles constructed using:"
)

print(
    "P_j = [T_j, M_j, D_j, R_j, C_j, Y_j]"
)

FEATURE PROFILE CONSTRUCTION
Datasets : 3
Features : 77
Complete profiles : 77
Incomplete profiles : 0


,dataset_id,feature,type_available,missingness_available,distribution_available,relevance_available,cardinality_available,dependency_available,complete_profile
0,adult_income,age,True,True,True,True,True,True,True
1,adult_income,workclass,True,True,True,True,True,True,True
2,adult_income,fnlwgt,True,True,True,True,True,True,True
3,adult_income,education,True,True,True,True,True,True,True
4,adult_income,education_num,True,True,True,True,True,True,True
5,adult_income,marital_status,True,True,True,True,True,True,True
6,adult_income,occupation,True,True,True,True,True,True,True
7,adult_income,relationship,True,True,True,True,True,True,True
8,adult_income,race,True,True,True,True,True,True,True
9,adult_income,sex,True,True,True,True,True,True,True



Feature profiles constructed using:
P_j = [T_j, M_j, D_j, R_j, C_j, Y_j]


In [38]:
# ============================================================
# 05.20 SAVE PROFILE ARTIFACTS
# ============================================================

import json
from pathlib import Path

print("=" * 90)
print("NOTEBOOK 05 — SAVING PROFILE ARTIFACTS")
print("=" * 90)

# ------------------------------------------------------------
# 1. PROFILE DIRECTORIES
# ------------------------------------------------------------

PROFILE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "profiles"
)

DATASET_PROFILE_DIR = (
    PROFILE_ROOT /
    "dataset"
)

FEATURE_PROFILE_DIR = (
    PROFILE_ROOT /
    "feature"
)

SUMMARY_PROFILE_DIR = (
    PROFILE_ROOT /
    "summary"
)

for path in [
    PROFILE_ROOT,
    DATASET_PROFILE_DIR,
    FEATURE_PROFILE_DIR,
    SUMMARY_PROFILE_DIR
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 2. REQUIRED OBJECT VALIDATION
# ------------------------------------------------------------

required_objects = {

    "DATASET_PROFILES":
        DATASET_PROFILES,

    "FEATURE_PROFILES":
        FEATURE_PROFILES,

    "DATASET_SIZE_DF":
        DATASET_SIZE_DF,

    "FEATURE_COUNT_DF":
        FEATURE_COUNT_DF,

    "TYPE_RATIO_DF":
        TYPE_RATIO_DF,

    "CLASS_DISTRIBUTION_DF":
        CLASS_DISTRIBUTION_DF,

    "MISSINGNESS_DF":
        MISSINGNESS_DF,

    "CARDINALITY_DF":
        CARDINALITY_DF,

    "DISTRIBUTION_DF":
        DISTRIBUTION_DF,

    "OUTLIER_DF":
        OUTLIER_DF,

    "CORRELATION_PROFILE":
        CORRELATION_PROFILE,

    "MUTUAL_INFORMATION_DF":
        MUTUAL_INFORMATION_DF,

    "DEPENDENCY_DF":
        DEPENDENCY_DF,

    "COMPUTATIONAL_SCALE_DF":
        COMPUTATIONAL_SCALE_DF,

    "FEATURE_TYPE_DF":
        FEATURE_TYPE_DF,

    "FEATURE_MISSINGNESS_DF":
        FEATURE_MISSINGNESS_DF,

    "FEATURE_DISTRIBUTION_DF":
        FEATURE_DISTRIBUTION_DF,

    "FEATURE_DEPENDENCY_DF":
        FEATURE_DEPENDENCY_DF
}

missing_objects = [
    name
    for name, obj in required_objects.items()
    if obj is None
]

if missing_objects:

    raise RuntimeError(
        "Notebook 05 cannot save profiles because "
        "the following required objects are missing:\n\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )

print("Required profile objects: AVAILABLE")

# ------------------------------------------------------------
# 3. SAVE CANONICAL DATASET PROFILE — JSON
# ------------------------------------------------------------

DATASET_PROFILE_JSON_PATH = (
    DATASET_PROFILE_DIR /
    "dataset_profiles.json"
)

with open(
    DATASET_PROFILE_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        DATASET_PROFILES,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# ------------------------------------------------------------
# 4. SAVE CANONICAL FEATURE PROFILE — JSON
# ------------------------------------------------------------

FEATURE_PROFILE_JSON_PATH = (
    FEATURE_PROFILE_DIR /
    "feature_profiles.json"
)

with open(
    FEATURE_PROFILE_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FEATURE_PROFILES,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# ------------------------------------------------------------
# 5. CREATE FLAT DATASET PROFILE TABLE
# ------------------------------------------------------------

DATASET_PROFILE_RECORDS = []

for dataset_id, profile in DATASET_PROFILES.items():

    size = profile.get(
        "size",
        {}
    )

    type_info = profile.get(
        "type",
        {}
    )

    missingness = profile.get(
        "missingness",
        {}
    )

    computational = profile.get(
        "computational_scale",
        {}
    )

    task = profile.get(
        "task",
        {}
    )

    DATASET_PROFILE_RECORDS.append({

        "dataset_id":
            dataset_id,

        "target":
            task.get("target"),

        "n_rows":
            size.get("rows"),

        "n_columns":
            size.get("columns"),

        "memory_mb":
            size.get("memory_mb"),

        "n_features":
            type_info.get("n_features"),

        "n_numeric_features":
            type_info.get("numeric_count"),

        "n_categorical_features":
            type_info.get("categorical_count"),

        "numeric_ratio":
            type_info.get("numeric_ratio"),

        "categorical_ratio":
            type_info.get("categorical_ratio"),

        "overall_missing_rate":
            missingness.get(
                "overall_feature_missing_rate"
            ),

        "cells":
            computational.get("cells"),

        "missing_cells":
            computational.get("missing_cells"),

        "missing_cell_rate":
            computational.get(
                "missing_cell_rate"
            ),

        "scale_category":
            computational.get(
                "scale_category"
            )
    })

DATASET_PROFILE_DF = pd.DataFrame(
    DATASET_PROFILE_RECORDS
)

DATASET_PROFILE_CSV_PATH = (
    DATASET_PROFILE_DIR /
    "dataset_profiles.csv"
)

DATASET_PROFILE_DF.to_csv(
    DATASET_PROFILE_CSV_PATH,
    index=False
)

# ------------------------------------------------------------
# 6. CREATE FLAT FEATURE PROFILE TABLE
#
# Only actual feature profiles are exported here.
# This is NOT MISSINGNESS_DF.
# ------------------------------------------------------------

FEATURE_PROFILE_RECORDS = []

for dataset_id, feature_dict in FEATURE_PROFILES.items():

    for feature, profile in feature_dict.items():

        record = {

            "dataset_id":
                dataset_id,

            "feature":
                feature
        }

        # ----------------------------------------------------
        # Type
        # ----------------------------------------------------

        type_info = profile.get(
            "T",
            {}
        )

        record.update({

            "feature_role":
                type_info.get(
                    "feature_role"
                ),

            "dtype":
                type_info.get(
                    "dtype"
                ),

            "is_numeric":
                type_info.get(
                    "is_numeric"
                ),

            "is_categorical":
                type_info.get(
                    "is_categorical"
                ),

            "is_target":
                type_info.get(
                    "is_target"
                )
        })

        # ----------------------------------------------------
        # Missingness
        # ----------------------------------------------------

        missing_info = profile.get(
            "M",
            {}
        )

        record.update({

            "missing_count":
                missing_info.get(
                    "missing_count"
                ),

            "missing_rate":
                missing_info.get(
                    "missing_rate"
                ),

            "missingness_category":
                missing_info.get(
                    "missingness_category"
                )
        })

        # ----------------------------------------------------
        # Distribution
        # ----------------------------------------------------

        distribution_info = profile.get(
            "D",
            {}
        )

        record.update({

            "distribution_type":
                distribution_info.get(
                    "distribution_type"
                ),

            "mean":
                distribution_info.get(
                    "mean"
                ),

            "median":
                distribution_info.get(
                    "median"
                ),

            "std":
                distribution_info.get(
                    "std"
                ),

            "skewness":
                distribution_info.get(
                    "skewness"
                ),

            "kurtosis":
                distribution_info.get(
                    "kurtosis"
                ),

            "distribution_shape":
                distribution_info.get(
                    "distribution_shape"
                ),

            "top_category_frequency":
                distribution_info.get(
                    "top_category_frequency"
                )
        })

        # ----------------------------------------------------
        # Cardinality
        # ----------------------------------------------------

        cardinality_info = profile.get(
            "C",
            {}
        )

        record.update({

            "unique_count":
                cardinality_info.get(
                    "unique_count"
                ),

            "cardinality_ratio":
                cardinality_info.get(
                    "cardinality_ratio"
                ),

            "is_constant":
                cardinality_info.get(
                    "is_constant"
                ),

            "is_high_cardinality":
                cardinality_info.get(
                    "is_high_cardinality"
                )
        })

        # ----------------------------------------------------
        # Predictive relevance
        # ----------------------------------------------------

        relevance_info = profile.get(
            "R",
            {}
        )

        record.update({

            "mutual_information":
                relevance_info.get(
                    "mutual_information"
                )
        })

        # ----------------------------------------------------
        # Dependencies
        # ----------------------------------------------------

        dependency_info = profile.get(
            "Y",
            {}
        )

        record.update({

            "max_abs_spearman":
                dependency_info.get(
                    "max_abs_spearman"
                ),

            "strongest_correlated_feature":
                dependency_info.get(
                    "strongest_correlated_feature"
                ),

            "dependency_strength":
                dependency_info.get(
                    "dependency_strength"
                )
        })

        FEATURE_PROFILE_RECORDS.append(
            record
        )

FEATURE_PROFILE_DF = pd.DataFrame(
    FEATURE_PROFILE_RECORDS
)

FEATURE_PROFILE_CSV_PATH = (
    FEATURE_PROFILE_DIR /
    "feature_profiles.csv"
)

FEATURE_PROFILE_DF.to_csv(
    FEATURE_PROFILE_CSV_PATH,
    index=False
)

# ------------------------------------------------------------
# 7. SAVE INDIVIDUAL SUPPORTING PROFILE TABLES
# ------------------------------------------------------------

SUPPORTING_PROFILE_TABLES = {

    "dataset_size":
        DATASET_SIZE_DF,

    "feature_count":
        FEATURE_COUNT_DF,

    "type_ratio":
        TYPE_RATIO_DF,

    "class_distribution":
        CLASS_DISTRIBUTION_DF,

    "missingness":
        MISSINGNESS_DF,

    "cardinality":
        CARDINALITY_DF,

    "distribution":
        DISTRIBUTION_DF,

    "outlier":
        OUTLIER_DF,

    "mutual_information":
        MUTUAL_INFORMATION_DF,

    "dependency":
        DEPENDENCY_DF,

    "computational_scale":
        COMPUTATIONAL_SCALE_DF,

    "feature_type":
        FEATURE_TYPE_DF,

    "feature_missingness":
        FEATURE_MISSINGNESS_DF,

    "feature_distribution":
        FEATURE_DISTRIBUTION_DF,

    "feature_dependency":
        FEATURE_DEPENDENCY_DF
}

SUPPORTING_PATHS = {}

for table_name, table_df in (
    SUPPORTING_PROFILE_TABLES.items()
):

    output_path = (
        SUMMARY_PROFILE_DIR /
        f"{table_name}.csv"
    )

    table_df.to_csv(
        output_path,
        index=False
    )

    SUPPORTING_PATHS[
        table_name
    ] = str(output_path)

# ------------------------------------------------------------
# 8. SAVE CORRELATION MATRICES
# ------------------------------------------------------------

CORRELATION_DIR = (
    SUMMARY_PROFILE_DIR /
    "correlations"
)

CORRELATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CORRELATION_PATHS = {}

for dataset_id, correlation_df in (
    CORRELATION_PROFILE.items()
):

    output_path = (
        CORRELATION_DIR /
        f"{dataset_id}_spearman.csv"
    )

    correlation_df.to_csv(
        output_path
    )

    CORRELATION_PATHS[
        dataset_id
    ] = str(output_path)

# ------------------------------------------------------------
# 9. SAVE PROFILE REGISTRY
# ------------------------------------------------------------

PROFILE_REGISTRY_RECORDS = [

    {
        "artifact":
            "dataset_profiles_json",

        "path":
            str(
                DATASET_PROFILE_JSON_PATH
            ),

        "format":
            "json",

        "rows":
            len(DATASET_PROFILES),

        "description":
            "Canonical structured dataset profiles"
    },

    {
        "artifact":
            "dataset_profiles_csv",

        "path":
            str(
                DATASET_PROFILE_CSV_PATH
            ),

        "format":
            "csv",

        "rows":
            int(
                DATASET_PROFILE_DF.shape[0]
            ),

        "description":
            "Flat dataset profile table for downstream notebooks"
    },

    {
        "artifact":
            "feature_profiles_json",

        "path":
            str(
                FEATURE_PROFILE_JSON_PATH
            ),

        "format":
            "json",

        "rows":
            sum(
                len(v)
                for v in FEATURE_PROFILES.values()
            ),

        "description":
            "Canonical structured feature profiles"
    },

    {
        "artifact":
            "feature_profiles_csv",

        "path":
            str(
                FEATURE_PROFILE_CSV_PATH
            ),

        "format":
            "csv",

        "rows":
            int(
                FEATURE_PROFILE_DF.shape[0]
            ),

        "description":
            "Flat feature profile table for downstream notebooks"
    }
]

PROFILE_REGISTRY_DF = pd.DataFrame(
    PROFILE_REGISTRY_RECORDS
)

PROFILE_REGISTRY_PATH = (
    PROFILE_ROOT /
    "profile_registry.csv"
)

PROFILE_REGISTRY_DF.to_csv(
    PROFILE_REGISTRY_PATH,
    index=False
)

# ------------------------------------------------------------
# 10. FINAL VALIDATION
# ------------------------------------------------------------

assert DATASET_PROFILE_JSON_PATH.exists()
assert DATASET_PROFILE_CSV_PATH.exists()

assert FEATURE_PROFILE_JSON_PATH.exists()
assert FEATURE_PROFILE_CSV_PATH.exists()

assert PROFILE_REGISTRY_PATH.exists()

assert set(
    DATASET_PROFILE_DF[
        "dataset_id"
    ]
) == set(DATASET_IDS)

assert {
    "dataset_id",
    "feature",
    "feature_role",
    "missing_rate",
    "unique_count",
    "mutual_information"
}.issubset(
    FEATURE_PROFILE_DF.columns
)

# ------------------------------------------------------------
# 11. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("=" * 90)
print("NOTEBOOK 05 — PROFILE ARTIFACTS SAVED")
print("=" * 90)

print(
    f"Dataset JSON : "
    f"{DATASET_PROFILE_JSON_PATH}"
)

print(
    f"Dataset CSV  : "
    f"{DATASET_PROFILE_CSV_PATH}"
)

print(
    f"Feature JSON : "
    f"{FEATURE_PROFILE_JSON_PATH}"
)

print(
    f"Feature CSV  : "
    f"{FEATURE_PROFILE_CSV_PATH}"
)

print(
    f"Registry     : "
    f"{PROFILE_REGISTRY_PATH}"
)

print()
print(
    f"Dataset profiles : "
    f"{DATASET_PROFILE_DF.shape}"
)

print(
    f"Feature profiles : "
    f"{FEATURE_PROFILE_DF.shape}"
)

print()
print("=" * 90)
print("NOTEBOOK 05 PROFILE SAVE COMPLETE")
print("=" * 90)

NOTEBOOK 05 — SAVING PROFILE ARTIFACTS
Required profile objects: AVAILABLE

NOTEBOOK 05 — PROFILE ARTIFACTS SAVED
Dataset JSON : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset/dataset_profiles.json
Dataset CSV  : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset/dataset_profiles.csv
Feature JSON : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature/feature_profiles.json
Feature CSV  : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature/feature_profiles.csv
Registry     : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/profile_registry.csv

Dataset profiles : (3, 15)
Feature profiles : (77, 26)

NOTEBOOK 05 PROFILE SAVE COMPLETE


In [39]:
# ============================================================
# 05.25 AIR-LLM CONTEXT CONSTRUCTION
# ============================================================

AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------
    # Validate dataset profile
    # --------------------------------------------------------

    if dataset_id not in DATASET_PROFILES:

        raise KeyError(
            f"Dataset profile not found for: "
            f"{dataset_id}"
        )

    # --------------------------------------------------------
    # Validate feature profiles
    # --------------------------------------------------------

    if dataset_id not in FEATURE_PROFILES:

        raise KeyError(
            f"Feature profiles not found for: "
            f"{dataset_id}"
        )

    dataset_profile = (
        DATASET_PROFILES[
            dataset_id
        ]
    )

    feature_profiles = (
        FEATURE_PROFILES[
            dataset_id
        ]
    )

    if not feature_profiles:

        raise RuntimeError(
            f"No feature profiles available "
            f"for dataset: {dataset_id}"
        )

    # --------------------------------------------------------
    # Construct AIR-LLM context
    # --------------------------------------------------------

    AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "target":
            TARGET_REGISTRY[
                dataset_id
            ],

        "dataset_profile":
            dataset_profile,

        "feature_profiles":
            feature_profiles
    }

# ============================================================
# VALIDATION
# ============================================================

assert set(
    AIR_LLM_CONTEXT.keys()
) == set(DATASET_IDS)

for dataset_id in DATASET_IDS:

    context = AIR_LLM_CONTEXT[
        dataset_id
    ]

    assert (
        "dataset_profile"
        in context
    )

    assert (
        "feature_profiles"
        in context
    )

    assert (
        "target"
        in context
    )

    assert isinstance(
        context[
            "feature_profiles"
        ],
        dict
    )

    assert len(
        context[
            "feature_profiles"
        ]
    ) > 0

# ============================================================
# SUMMARY
# ============================================================

print("=" * 90)
print("AIR-LLM CONTEXT CONSTRUCTION COMPLETE")
print("=" * 90)

for dataset_id in DATASET_IDS:

    context = AIR_LLM_CONTEXT[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} | "
        f"target: "
        f"{context['target']:15s} | "
        f"features: "
        f"{len(context['feature_profiles']):3d}"
    )

print()
print(
    "AIR-LLM context contains:"
)

print(
    "  • Dataset-level profile"
)

print(
    "  • Feature-level profiles"
)

print(
    "  • Target/task information"
)

print()
print(
    "Context is ready for Notebook 06."
)

AIR-LLM CONTEXT CONSTRUCTION COMPLETE
adult_income         | target: income          | features:  14
bank_marketing       | target: y               | features:  16
diabetes_130us       | target: readmitted      | features:  47

AIR-LLM context contains:
  • Dataset-level profile
  • Feature-level profiles
  • Target/task information

Context is ready for Notebook 06.


In [40]:
# ============================================================
# 05.26 COMPACT AIR-LLM CONTEXT
# ============================================================

COMPACT_AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------
    # Retrieve canonical profiles
    # --------------------------------------------------------

    dataset_profile = DATASET_PROFILES[
        dataset_id
    ]

    feature_profiles = FEATURE_PROFILES[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # --------------------------------------------------------
    # Select only incomplete features
    # --------------------------------------------------------

    incomplete_features = {}

    for feature, profile in feature_profiles.items():

        missing_rate = (
            profile
            .get("M", {})
            .get("missing_rate", 0.0)
        )

        if (
            pd.notna(missing_rate)
            and float(missing_rate) > 0
            and feature != target
        ):

            incomplete_features[
                feature
            ] = {

                "T":
                    profile.get(
                        "T",
                        {}
                    ),

                "M":
                    profile.get(
                        "M",
                        {}
                    ),

                "D":
                    profile.get(
                        "D",
                        {}
                    ),

                "R":
                    profile.get(
                        "R",
                        {}
                    ),

                "C":
                    profile.get(
                        "C",
                        {}
                    ),

                "Y":
                    profile.get(
                        "Y",
                        {}
                    )
            }

    # --------------------------------------------------------
    # Construct compact dataset context
    # --------------------------------------------------------

    COMPACT_AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "target":
            target,

        "dataset": {

            "size":
                dataset_profile.get(
                    "size",
                    {}
                ),

            "type":
                dataset_profile.get(
                    "type",
                    {}
                ),

            "distribution":
                dataset_profile.get(
                    "distribution",
                    {}
                ),

            "dependency":
                dataset_profile.get(
                    "dependency",
                    {}
                ),

            "missingness":
                dataset_profile.get(
                    "missingness",
                    {}
                ),

            "task":
                dataset_profile.get(
                    "task",
                    {}
                ),

            "computational_scale":
                dataset_profile.get(
                    "computational_scale",
                    {}
                )
        },

        "incomplete_feature_count":
            len(incomplete_features),

        "incomplete_features":
            incomplete_features
    }

# ============================================================
# VALIDATION
# ============================================================

assert set(
    COMPACT_AIR_LLM_CONTEXT.keys()
) == set(DATASET_IDS)

for dataset_id in DATASET_IDS:

    context = COMPACT_AIR_LLM_CONTEXT[
        dataset_id
    ]

    assert (
        "dataset"
        in context
    )

    assert (
        "incomplete_features"
        in context
    )

    assert (
        "incomplete_feature_count"
        in context
    )

    assert (
        context[
            "incomplete_feature_count"
        ]
        ==
        len(
            context[
                "incomplete_features"
            ]
        )
    )

# ============================================================
# SUMMARY
# ============================================================

print("=" * 90)
print("COMPACT AIR-LLM CONTEXT CONSTRUCTED")
print("=" * 90)

for dataset_id in DATASET_IDS:

    context = COMPACT_AIR_LLM_CONTEXT[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} | "
        f"incomplete features: "
        f"{context['incomplete_feature_count']}"
    )

print()
print(
    "Compact context contains:"
)

print(
    "  • Dataset size"
)

print(
    "  • Feature-type composition"
)

print(
    "  • Distribution information"
)

print(
    "  • Feature dependencies"
)

print(
    "  • Dataset missingness"
)

print(
    "  • Task / target information"
)

print(
    "  • Computational scale"
)

print(
    "  • Complete profiles for incomplete features"
)

COMPACT AIR-LLM CONTEXT CONSTRUCTED
adult_income         | incomplete features: 0
bank_marketing       | incomplete features: 0
diabetes_130us       | incomplete features: 9

Compact context contains:
  • Dataset size
  • Feature-type composition
  • Distribution information
  • Feature dependencies
  • Dataset missingness
  • Task / target information
  • Computational scale
  • Complete profiles for incomplete features


In [42]:
import json
from pathlib import Path

# ============================================================
# 05.27 SAVE PROFILING ARTIFACTS
# ============================================================

# ------------------------------------------------------------
# Dataset-level tabular artifacts
# ------------------------------------------------------------

DATASET_SIZE_DF.to_csv(
    DATASET_PROFILE_DIR /
    "dataset_size.csv",
    index=False
)

FEATURE_COUNT_DF.to_csv(
    DATASET_PROFILE_DIR /
    "feature_counts.csv",
    index=False
)

TYPE_RATIO_DF.to_csv(
    DATASET_PROFILE_DIR /
    "feature_type_ratios.csv",
    index=False
)

CLASS_DISTRIBUTION_DF.to_csv(
    DATASET_PROFILE_DIR /
    "class_distribution.csv",
    index=False
)

COMPUTATIONAL_SCALE_DF.to_csv(
    DATASET_PROFILE_DIR /
    "computational_scale.csv",
    index=False
)

# ------------------------------------------------------------
# Feature-level statistical artifacts
# ------------------------------------------------------------

MISSINGNESS_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "missingness_statistics.csv",
    index=False
)

CARDINALITY_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "cardinality_statistics.csv",
    index=False
)

DISTRIBUTION_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "distribution_statistics.csv",
    index=False
)

OUTLIER_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "outlier_statistics.csv",
    index=False
)

DEPENDENCY_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_dependencies.csv",
    index=False
)

MUTUAL_INFORMATION_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "mutual_information.csv",
    index=False
)

# ------------------------------------------------------------
# Correlation matrices
# ------------------------------------------------------------

CORRELATION_DIR = (
    FEATURE_PROFILE_DIR /
    "correlations"
)

CORRELATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for dataset_id, correlation_matrix in (
    CORRELATION_PROFILE.items()
):

    if (
        correlation_matrix is not None
        and not correlation_matrix.empty
    ):

        correlation_matrix.to_csv(
            CORRELATION_DIR /
            f"{dataset_id}_spearman.csv"
        )

# ------------------------------------------------------------
# Feature profiling artifacts
# ------------------------------------------------------------

FEATURE_TYPE_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_types.csv",
    index=False
)

FEATURE_MISSINGNESS_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_missingness.csv",
    index=False
)

FEATURE_DISTRIBUTION_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_distributions.csv",
    index=False
)

FEATURE_DEPENDENCY_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_dependencies_enriched.csv",
    index=False
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

required_artifacts = [

    DATASET_PROFILE_DIR /
    "dataset_size.csv",

    DATASET_PROFILE_DIR /
    "feature_counts.csv",

    DATASET_PROFILE_DIR /
    "feature_type_ratios.csv",

    DATASET_PROFILE_DIR /
    "class_distribution.csv",

    DATASET_PROFILE_DIR /
    "computational_scale.csv",

    FEATURE_PROFILE_DIR /
    "missingness_statistics.csv",

    FEATURE_PROFILE_DIR /
    "cardinality_statistics.csv",

    FEATURE_PROFILE_DIR /
    "distribution_statistics.csv",

    FEATURE_PROFILE_DIR /
    "outlier_statistics.csv",

    FEATURE_PROFILE_DIR /
    "feature_dependencies.csv",

    FEATURE_PROFILE_DIR /
    "mutual_information.csv",

    FEATURE_PROFILE_DIR /
    "feature_types.csv",

    FEATURE_PROFILE_DIR /
    "feature_missingness.csv",

    FEATURE_PROFILE_DIR /
    "feature_distributions.csv",

    FEATURE_PROFILE_DIR /
    "feature_dependencies_enriched.csv"
]

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

if missing_artifacts:

    raise RuntimeError(
        "Profiling artifact save failed:\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_artifacts
        )
    )

print("=" * 90)
print("NOTEBOOK 05 — PROFILING ARTIFACTS SAVED")
print("=" * 90)

print(
    f"Dataset artifacts : "
    f"{sum(1 for p in required_artifacts if DATASET_PROFILE_DIR in p.parents)}"
)

print(
    f"Feature artifacts : "
    f"{len(required_artifacts)} total validated"
)

print(
    f"Correlation files : "
    f"{len(CORRELATION_PROFILE)} datasets"
)

print()
print(
    "All profiling artifacts verified successfully."
)

NOTEBOOK 05 — PROFILING ARTIFACTS SAVED
Dataset artifacts : 5
Feature artifacts : 15 total validated
Correlation files : 3 datasets

All profiling artifacts verified successfully.


In [43]:
# ============================================================
# 05.29 SAVE CANONICAL AIR-LLM JSON CONTEXT
# ============================================================

import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# JSON-SAFE CONVERSION
# ------------------------------------------------------------

def make_json_safe(obj):

    if isinstance(obj, dict):

        return {
            str(key):
                make_json_safe(value)
            for key, value in obj.items()
        }

    if isinstance(obj, (list, tuple)):

        return [
            make_json_safe(value)
            for value in obj
        ]

    if isinstance(obj, np.ndarray):

        return [
            make_json_safe(value)
            for value in obj.tolist()
        ]

    if isinstance(obj, np.integer):

        return int(obj)

    if isinstance(obj, np.floating):

        value = float(obj)

        return (
            None
            if not np.isfinite(value)
            else value
        )

    if isinstance(obj, np.bool_):

        return bool(obj)

    if isinstance(obj, float):

        return (
            None
            if not np.isfinite(obj)
            else obj
        )

    if isinstance(obj, pd.Timestamp):

        return obj.isoformat()

    if obj is pd.NA:

        return None

    return obj


# ------------------------------------------------------------
# CANONICAL PROFILE PATHS
# ------------------------------------------------------------

DATASET_CONTEXT_PATH = (
    PROFILE_ROOT /
    "dataset_profiles.json"
)

FEATURE_CONTEXT_PATH = (
    PROFILE_ROOT /
    "feature_profiles.json"
)

AIR_LLM_CONTEXT_PATH = (
    PROFILE_ROOT /
    "air_llm_context.json"
)

COMPACT_CONTEXT_PATH = (
    PROFILE_ROOT /
    "compact_air_llm_context.json"
)


# ------------------------------------------------------------
# SAVE DATASET PROFILES
# ------------------------------------------------------------

with open(
    DATASET_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            DATASET_PROFILES
        ),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# SAVE FEATURE PROFILES
# ------------------------------------------------------------

with open(
    FEATURE_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            FEATURE_PROFILES
        ),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# SAVE COMPLETE AIR-LLM CONTEXT
# ------------------------------------------------------------

with open(
    AIR_LLM_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            AIR_LLM_CONTEXT
        ),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# SAVE COMPACT AIR-LLM CONTEXT
# ------------------------------------------------------------

with open(
    COMPACT_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            COMPACT_AIR_LLM_CONTEXT
        ),
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

JSON_CONTEXT_PATHS = [

    DATASET_CONTEXT_PATH,
    FEATURE_CONTEXT_PATH,
    AIR_LLM_CONTEXT_PATH,
    COMPACT_CONTEXT_PATH
]

missing_json_files = [
    str(path)
    for path in JSON_CONTEXT_PATHS
    if not path.exists()
]

if missing_json_files:

    raise RuntimeError(
        "AIR-LLM JSON context files were not "
        "created successfully:\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_json_files
        )
    )


# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("=" * 90)
print("AIR-LLM JSON CONTEXT FILES SAVED")
print("=" * 90)

for path in JSON_CONTEXT_PATHS:

    print(
        f"PASS | {path}"
    )

print()
print(
    f"Datasets : {len(DATASET_PROFILES)}"
)

print(
    f"Features : "
    f"{sum(len(v) for v in FEATURE_PROFILES.values())}"
)

print(
    "Complete and compact AIR-LLM contexts "
    "are available for downstream notebooks."
)

AIR-LLM JSON CONTEXT FILES SAVED
PASS | /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset_profiles.json
PASS | /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature_profiles.json
PASS | /content/drive/MyDrive/AIR_LLM_Research/results/profiles/air_llm_context.json
PASS | /content/drive/MyDrive/AIR_LLM_Research/results/profiles/compact_air_llm_context.json

Datasets : 3
Features : 77
Complete and compact AIR-LLM contexts are available for downstream notebooks.


In [45]:
# ============================================================
# 05.29 FINAL PROFILE CONSTRUCTION + SAVE
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 90)
print("NOTEBOOK 05 — FINAL PROFILE CONSTRUCTION AND SAVE")
print("=" * 90)

# ------------------------------------------------------------
# 1. CANONICAL OUTPUT DIRECTORIES
# ------------------------------------------------------------

PROFILE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "profiles"
)

DATASET_PROFILE_DIR = (
    PROFILE_ROOT /
    "dataset"
)

FEATURE_PROFILE_DIR = (
    PROFILE_ROOT /
    "feature"
)

PROFILE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_PROFILE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. BUILD CANONICAL DATASET PROFILE TABLE
# ------------------------------------------------------------

DATASET_PROFILE_DF = (
    DATASET_SIZE_DF.copy()
)

DATASET_PROFILE_DF = (
    DATASET_PROFILE_DF
    .merge(
        FEATURE_COUNT_DF,
        on="dataset_id",
        how="left"
    )
    .merge(
        TYPE_RATIO_DF,
        on="dataset_id",
        how="left"
    )
    .merge(
        COMPUTATIONAL_SCALE_DF[
            [
                "dataset_id",
                "missing_cells",
                "missing_cell_rate",
                "scale_category"
            ]
        ],
        on="dataset_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 3. VALIDATE DATASET PROFILE
# ------------------------------------------------------------

assert set(
    DATASET_PROFILE_DF["dataset_id"]
) == set(DATASET_IDS), (
    "Dataset profile does not contain "
    "all required datasets."
)

assert (
    DATASET_PROFILE_DF["dataset_id"]
    .duplicated()
    .sum()
    == 0
), (
    "Dataset profile contains duplicate "
    "dataset IDs."
)

# ------------------------------------------------------------
# 4. BUILD CANONICAL FEATURE PROFILE TABLE
# ------------------------------------------------------------

FEATURE_PROFILE_DF = (
    FEATURE_TYPE_DF.copy()
)

FEATURE_PROFILE_DF = (
    FEATURE_PROFILE_DF
    .merge(
        FEATURE_MISSINGNESS_DF[
            [
                "dataset_id",
                "feature",
                "missing_count",
                "missing_rate",
                "missingness_category"
            ]
        ],
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
    .merge(
        FEATURE_DISTRIBUTION_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
    .merge(
        CARDINALITY_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
    .merge(
        FEATURE_DEPENDENCY_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
    .merge(
        MUTUAL_INFORMATION_DF,
        on=[
            "dataset_id",
            "feature"
        ],
        how="left"
    )
)

# ------------------------------------------------------------
# 5. VALIDATE FEATURE PROFILE
# ------------------------------------------------------------

assert (
    "dataset_id"
    in FEATURE_PROFILE_DF.columns
)

assert (
    "feature"
    in FEATURE_PROFILE_DF.columns
)

assert (
    set(
        FEATURE_PROFILE_DF[
            "dataset_id"
        ]
    )
    == set(DATASET_IDS)
), (
    "Feature profile does not contain "
    "all required datasets."
)

assert not (
    FEATURE_PROFILE_DF[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .duplicated()
    .any()
), (
    "Feature profile contains duplicate "
    "dataset-feature combinations."
)

# ------------------------------------------------------------
# 6. CONSTRUCT AIR-LLM CONTEXT
# ------------------------------------------------------------

AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    dataset_profile = (
        DATASET_PROFILES[
            dataset_id
        ]
    )

    feature_profiles = (
        FEATURE_PROFILES[
            dataset_id
        ]
    )

    AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset_profile":
            dataset_profile,

        "feature_profiles":
            feature_profiles
    }

# ------------------------------------------------------------
# 7. COMPACT AIR-LLM CONTEXT
# ------------------------------------------------------------

COMPACT_AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    dataset_profile = (
        DATASET_PROFILES[
            dataset_id
        ]
    )

    feature_profiles = (
        FEATURE_PROFILES[
            dataset_id
        ]
    )

    incomplete_features = {}

    for feature, profile in (
        feature_profiles.items()
    ):

        missing_rate = (
            profile
            .get("M", {})
            .get("missing_rate", 0.0)
        )

        if pd.notna(
            missing_rate
        ) and float(
            missing_rate
        ) > 0:

            incomplete_features[
                feature
            ] = profile

    COMPACT_AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset":
            dataset_profile,

        "incomplete_features":
            incomplete_features
    }

# ------------------------------------------------------------
# 8. SAVE CANONICAL DATASET PROFILE
# ------------------------------------------------------------

DATASET_PROFILE_PATH = (
    DATASET_PROFILE_DIR /
    "dataset_profiles.csv"
)

DATASET_PROFILE_DF.to_csv(
    DATASET_PROFILE_PATH,
    index=False
)

# ------------------------------------------------------------
# 9. SAVE CANONICAL FEATURE PROFILE
# ------------------------------------------------------------

FEATURE_PROFILE_PATH = (
    FEATURE_PROFILE_DIR /
    "feature_profiles.csv"
)

FEATURE_PROFILE_DF.to_csv(
    FEATURE_PROFILE_PATH,
    index=False
)

# ------------------------------------------------------------
# 10. JSON-SAFE CONVERSION
# ------------------------------------------------------------

def make_json_safe(obj):

    if isinstance(obj, dict):

        return {
            str(k):
                make_json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):

        return [
            make_json_safe(v)
            for v in obj
        ]

    if isinstance(obj, tuple):

        return [
            make_json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):

        return int(obj)

    if isinstance(obj, np.floating):

        value = float(obj)

        return (
            None
            if not np.isfinite(value)
            else value
        )

    if isinstance(obj, np.bool_):

        return bool(obj)

    if isinstance(obj, float):

        return (
            None
            if not np.isfinite(obj)
            else obj
        )

    return obj

# ------------------------------------------------------------
# 11. SAVE AIR-LLM CONTEXT
# ------------------------------------------------------------

AIR_LLM_CONTEXT_PATH = (
    PROFILE_ROOT /
    "air_llm_context.json"
)

with open(
    AIR_LLM_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            COMPACT_AIR_LLM_CONTEXT
        ),
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 12. SAVE COMPONENT TABLES
# ------------------------------------------------------------

DATASET_COMPONENTS = {

    "dataset_size.csv":
        DATASET_SIZE_DF,

    "feature_counts.csv":
        FEATURE_COUNT_DF,

    "feature_type_ratios.csv":
        TYPE_RATIO_DF,

    "class_distribution.csv":
        CLASS_DISTRIBUTION_DF,

    "missingness_statistics.csv":
        MISSINGNESS_DF,

    "cardinality_statistics.csv":
        CARDINALITY_DF,

    "distribution_statistics.csv":
        DISTRIBUTION_DF,

    "outlier_statistics.csv":
        OUTLIER_DF,

    "computational_scale.csv":
        COMPUTATIONAL_SCALE_DF,

    "feature_dependencies.csv":
        DEPENDENCY_DF
}

for filename, table in (
    DATASET_COMPONENTS.items()
):

    table.to_csv(
        DATASET_PROFILE_DIR /
        filename,
        index=False
    )

FEATURE_PREDICTIVE_RELEVANCE_DF = (
    MUTUAL_INFORMATION_DF.copy()
)

FEATURE_COMPONENTS = {

    "feature_types.csv":
        FEATURE_TYPE_DF,

    "feature_missingness.csv":
        FEATURE_MISSINGNESS_DF,

    "feature_distributions.csv":
        FEATURE_DISTRIBUTION_DF,

    "feature_cardinality.csv":
        CARDINALITY_DF,

    "feature_dependencies.csv":
        FEATURE_DEPENDENCY_DF,

    "feature_predictive_relevance.csv":
        FEATURE_PREDICTIVE_RELEVANCE_DF
}

for filename, table in (
    FEATURE_COMPONENTS.items()
):

    table.to_csv(
        FEATURE_PROFILE_DIR /
        filename,
        index=False
    )

# ------------------------------------------------------------
# 13. CANONICAL PROFILE REGISTRY
# ------------------------------------------------------------

PROFILE_REGISTRY_DF = pd.DataFrame([

    {
        "artifact":
            "dataset_profiles",

        "path":
            str(DATASET_PROFILE_PATH),

        "format":
            "csv",

        "rows":
            int(
                DATASET_PROFILE_DF.shape[0]
            ),

        "columns":
            int(
                DATASET_PROFILE_DF.shape[1]
            )
    },

    {
        "artifact":
            "feature_profiles",

        "path":
            str(FEATURE_PROFILE_PATH),

        "format":
            "csv",

        "rows":
            int(
                FEATURE_PROFILE_DF.shape[0]
            ),

        "columns":
            int(
                FEATURE_PROFILE_DF.shape[1]
            )
    },

    {
        "artifact":
            "air_llm_context",

        "path":
            str(AIR_LLM_CONTEXT_PATH),

        "format":
            "json",

        "rows":
            None,

        "columns":
            None
    }
])

PROFILE_REGISTRY_PATH = (
    PROFILE_ROOT /
    "profile_registry.csv"
)

PROFILE_REGISTRY_DF.to_csv(
    PROFILE_REGISTRY_PATH,
    index=False
)

# ------------------------------------------------------------
# 14. VERIFY REGISTRY IMMEDIATELY
# ------------------------------------------------------------

registry_artifacts = set(
    PROFILE_REGISTRY_DF[
        "artifact"
    ].astype(str)
)

required_artifacts = {
    "dataset_profiles",
    "feature_profiles",
    "air_llm_context"
}

assert required_artifacts.issubset(
    registry_artifacts
), (
    "Canonical profile registry "
    "was not constructed correctly."
)

# ------------------------------------------------------------
# 15. FINAL SAVE SUMMARY
# ------------------------------------------------------------

print()
print("=" * 90)
print("NOTEBOOK 05 — PROFILE ARTIFACTS SAVED")
print("=" * 90)

print(
    f"Dataset profile : "
    f"{DATASET_PROFILE_PATH}"
)

print(
    f"Feature profile : "
    f"{FEATURE_PROFILE_PATH}"
)

print(
    f"AIR-LLM context : "
    f"{AIR_LLM_CONTEXT_PATH}"
)

print(
    f"Registry        : "
    f"{PROFILE_REGISTRY_PATH}"
)

print()
print(
    f"Dataset profile exists : "
    f"{DATASET_PROFILE_PATH.exists()}"
)

print(
    f"Feature profile exists : "
    f"{FEATURE_PROFILE_PATH.exists()}"
)

print(
    f"Context exists         : "
    f"{AIR_LLM_CONTEXT_PATH.exists()}"
)

print(
    f"Registry exists        : "
    f"{PROFILE_REGISTRY_PATH.exists()}"
)

print()
print("=" * 90)
print("PROFILE SAVE COMPLETE")
print("=" * 90)

NOTEBOOK 05 — FINAL PROFILE CONSTRUCTION AND SAVE

NOTEBOOK 05 — PROFILE ARTIFACTS SAVED
Dataset profile : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/dataset/dataset_profiles.csv
Feature profile : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/feature/feature_profiles.csv
AIR-LLM context : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/air_llm_context.json
Registry        : /content/drive/MyDrive/AIR_LLM_Research/results/profiles/profile_registry.csv

Dataset profile exists : True
Feature profile exists : True
Context exists         : True
Registry exists        : True

PROFILE SAVE COMPLETE
